Read each paper's publication text plus the paper's .raw filenames, extract allowed SDRF metadat, and turn that into a valid submission table. Goal is not perfect row reconstruction, it is to recover the correct set of metadata values per paper and column as cleanly as possible. The dataset gives training paper JSONs, gold SDRFs, GPT-made training extracts, test paper JSONs, and a sample submission format.

Test_PubText/ is the test stuff for inference.
Training_GPT_Extract/ is GPT-generated metadata extracts for the training papers made using the baseline prompt, these are useful as baseline / weak teacher / candidate source
Training_PubText/ is the training papers as /json publication text files. These are the inputs we learn patterns from.
Training_SDRFs/ is the gold SDRF annotations fro the training set. This is the real target we compare against locally.
BaselinePrompt.txt is the official baseline extraction prompt, it defines what metadata categories are allowed, what manuscript sections to read, how to use .raw filenames, how to assign metadata per file, and what the output format should look like.

In [24]:
# ================================
# CELL 1: SETUP + LOADERS + TEXT VIEWS + GPT EXTRACT LOADER
# ================================

from pathlib import Path
import json
import re
import unicodedata
from collections import Counter, defaultdict
from typing import Dict, List, Tuple, Iterable, Optional, Any

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from sklearn.cluster import AgglomerativeClustering
import difflib

DATA_DIR = Path("Data")
TRAIN_PUB_DIR = DATA_DIR / "Training_PubText"
TRAIN_SDRF_DIR = DATA_DIR / "Training_SDRFs"
TRAIN_GPT_DIR = DATA_DIR / "Training_GPT_Extract"
TEST_PUB_DIR = DATA_DIR / "Test_PubText"
SAMPLE_SUB_PATH = DATA_DIR / "SampleSubmission.csv"

DEFAULT_FILL = "Not Applicable"
NA_VALUES = {"", "nan", "none", "null", "na", "not applicable", "not available"}
SYSTEM_COLUMNS = {"ID", "PXD", "Source Name", "Assay Name", "Raw Data File", "Data File"}

sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
SUB_COLUMNS = sample_sub.columns.tolist()

def clean_text(x) -> str:
    if x is None:
        return ""
    return re.sub(r"\s+", " ", str(x)).strip()

def base_annotation(col: str) -> str:
    col = str(col).strip()
    return col.split(".", 1)[0].strip()

def normalize_annotation_name(name: str) -> str:
    s = str(name).strip()
    s = re.sub(r"\.\d+$", "", s)
    s = re.sub(r"\s+", "", s)
    return s.lower()

BASE_TO_SUBCOLS = defaultdict(list)
BASE_META_COLS = []
_seen_base = set()

for col in SUB_COLUMNS:
    base = base_annotation(col)
    if col not in SYSTEM_COLUMNS and base not in SYSTEM_COLUMNS:
        BASE_TO_SUBCOLS[base].append(col)
        if base not in _seen_base:
            _seen_base.add(base)
            BASE_META_COLS.append(base)

META_COL_LOOKUP = {
    normalize_annotation_name(base): base
    for base in BASE_META_COLS
}

def load_sdrf(path: str | Path) -> pd.DataFrame:
    path = Path(path)

    try:
        df = pd.read_csv(path, sep=None, engine="python")
    except Exception:
        df = pd.read_csv(path)

    if df.shape[1] == 1:
        raw = pd.read_csv(path, header=None)
        one_col = raw.iloc[:, 0].astype(str)

        if one_col.str.contains("\t").any():
            split_rows = one_col.str.split("\t", expand=True)
        elif one_col.str.contains(",").any():
            split_rows = one_col.str.split(",", expand=True)
        else:
            return df

        header = split_rows.iloc[0].tolist()
        data = split_rows.iloc[1:].reset_index(drop=True)
        data.columns = header
        df = data

    return df

def load_any_table(path: str | Path) -> pd.DataFrame:
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix in [".tsv", ".txt"]:
        try:
            return pd.read_csv(path, sep="\t")
        except Exception:
            return pd.read_csv(path, sep=None, engine="python")

    if suffix in [".csv"]:
        try:
            return pd.read_csv(path)
        except Exception:
            return pd.read_csv(path, sep=None, engine="python")

    return pd.read_csv(path, sep=None, engine="python")

def load_pub_json(pxd: str, split: str = "train") -> dict:
    folder = TRAIN_PUB_DIR if split == "train" else TEST_PUB_DIR
    path = folder / f"{pxd}_PubText.json"
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def load_gold_sdrf(pxd: str) -> pd.DataFrame:
    path = TRAIN_SDRF_DIR / f"Harmonized_{pxd}.csv"
    return load_sdrf(path)

def _walk_json_text(obj: Any, path: str = "root"):
    if isinstance(obj, dict):
        for k, v in obj.items():
            yield from _walk_json_text(v, f"{path}.{k}")
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            yield from _walk_json_text(v, f"{path}[{i}]")
    else:
        txt = clean_text(obj)
        if txt:
            yield path, txt

def _unique_chunks(chunks: Iterable[str]) -> List[str]:
    out = []
    seen = set()
    for ch in chunks:
        s = clean_text(ch)
        if not s:
            continue
        k = s.lower()
        if k not in seen:
            seen.add(k)
            out.append(s)
    return out

def get_section_text(pub_json: dict, key: str) -> str:
    val = pub_json.get(key, "")
    if isinstance(val, str):
        return clean_text(val)
    if isinstance(val, list):
        return clean_text(" ".join(str(v) for v in val))
    if isinstance(val, dict):
        return clean_text(" ".join(f"{k}: {v}" for k, v in val.items()))
    return clean_text(val)

RAW_FILE_RE = re.compile(r'(?i)\b[^\s"\'<>(),;]+\.raw\b')

def extract_raw_files(pub_json: dict) -> List[str]:
    vals = []

    direct = pub_json.get("Raw Data Files", None)
    if isinstance(direct, list):
        vals.extend([clean_text(x) for x in direct if clean_text(x)])
    elif isinstance(direct, str):
        vals.extend(RAW_FILE_RE.findall(direct))

    for _, txt in _walk_json_text(pub_json):
        vals.extend(RAW_FILE_RE.findall(txt))

    out = []
    seen = set()
    for v in vals:
        vv = Path(str(v)).name
        if not vv:
            continue
        k = vv.lower()
        if k not in seen:
            seen.add(k)
            out.append(vv)
    return out

SAMPLE_KEYWORDS = [
    "sample", "samples", "patient", "patients", "subject", "subjects", "clinical",
    "cohort", "tissue", "tumor", "tumour", "biopsy", "specimen", "plasma", "serum",
    "cell", "cells", "cell line", "cell-line", "organ", "disease", "control", "treated"
]

METHOD_KEYWORDS = [
    "method", "methods", "experimental", "sample preparation", "mass spectrometry",
    "proteomics", "lc-ms", "lc ms", "chromatography", "database search"
]

RESULT_KEYWORDS = [
    "result", "results", "discussion", "findings"
]

CAPTION_KEYWORDS = [
    "figure", "table", "caption", "legend", "supplementary figure", "supplementary table"
]

def get_text_views(pub_json: dict) -> Dict[str, str]:
    title = get_section_text(pub_json, "TITLE")
    abstract = get_section_text(pub_json, "ABSTRACT")
    methods_direct = get_section_text(pub_json, "METHODS")

    all_chunks = []
    method_chunks = []
    sample_chunks = []
    result_chunks = []
    caption_chunks = []

    for path, txt in _walk_json_text(pub_json):
        p = path.lower()

        if "raw data file" in p:
            continue

        all_chunks.append(txt)

        if any(k in p for k in METHOD_KEYWORDS):
            method_chunks.append(txt)
        if any(k in p for k in SAMPLE_KEYWORDS):
            sample_chunks.append(txt)
        if any(k in p for k in RESULT_KEYWORDS):
            result_chunks.append(txt)
        if any(k in p for k in CAPTION_KEYWORDS):
            caption_chunks.append(txt)

    all_chunks = _unique_chunks(all_chunks)
    method_chunks = _unique_chunks([methods_direct] + method_chunks)
    sample_chunks = _unique_chunks(sample_chunks)
    result_chunks = _unique_chunks(result_chunks)
    caption_chunks = _unique_chunks(caption_chunks)

    core_chunks = _unique_chunks([title, abstract] + method_chunks)

    return {
        "title": title,
        "abstract": abstract,
        "methods_text": " ".join(method_chunks),
        "sample_text": " ".join(sample_chunks),
        "results_text": " ".join(result_chunks),
        "caption_text": " ".join(caption_chunks),
        "core_text": " ".join(core_chunks),
        "full_text": " ".join(all_chunks),
    }

def make_blank_row() -> Dict[str, str]:
    return {col: DEFAULT_FILL for col in SUB_COLUMNS}

def is_real_value_basic(x) -> bool:
    return clean_text(x).lower() not in NA_VALUES

def find_matching_file(folder: Path, pxd: str) -> Optional[Path]:
    if not folder.exists():
        return None

    exact = list(folder.glob(f"*{pxd}*"))
    if exact:
        exact = sorted(exact, key=lambda p: (len(p.name), p.name))
        return exact[0]
    return None

def _append_value_pair(out: Dict[str, List[str]], ann: str, value: Any):
    if ann is None:
        return

    ann_key = META_COL_LOOKUP.get(normalize_annotation_name(ann), None)
    if ann_key is None:
        return

    if isinstance(value, list):
        vals = value
    else:
        vals = [value]

    for v in vals:
        s = clean_text(v)
        if not s or not is_real_value_basic(s):
            continue
        out[ann_key].append(s)

def _extract_pairs_from_obj(obj: Any, out: Dict[str, List[str]]):
    if isinstance(obj, dict):
        lower_keys = {str(k).lower(): k for k in obj.keys()}

        ann_key = None
        val_key = None

        ann_candidates = [
            "annotationtype", "annotation_type", "annotation", "column",
            "field", "sdrf_column", "metadatatype", "metadata_type"
        ]
        val_candidates = [
            "value", "values", "prediction", "predicted_value",
            "answer", "text", "span", "matched_text"
        ]

        for k in ann_candidates:
            if k in lower_keys:
                ann_key = lower_keys[k]
                break

        for k in val_candidates:
            if k in lower_keys:
                val_key = lower_keys[k]
                break

        if ann_key is not None and val_key is not None:
            _append_value_pair(out, obj[ann_key], obj[val_key])

        for k, v in obj.items():
            kk = str(k)
            if normalize_annotation_name(kk) in META_COL_LOOKUP:
                _append_value_pair(out, kk, v)
            _extract_pairs_from_obj(v, out)

    elif isinstance(obj, list):
        for x in obj:
            _extract_pairs_from_obj(x, out)

def load_gpt_extract_as_value_dict(pxd: str) -> Dict[str, List[str]]:
    out = defaultdict(list)
    path = find_matching_file(TRAIN_GPT_DIR, pxd)

    if path is None:
        return {}

    suffix = path.suffix.lower()

    try:
        if suffix == ".json":
            with open(path, "r", encoding="utf-8") as f:
                obj = json.load(f)
            _extract_pairs_from_obj(obj, out)

        elif suffix in [".csv", ".tsv", ".txt"]:
            df = load_any_table(path)

            # Case 1: actual SDRF-style columns
            for col in df.columns:
                if normalize_annotation_name(col) in META_COL_LOOKUP:
                    _append_value_pair(out, col, df[col].dropna().astype(str).tolist())

            # Case 2: annotation/value style table
            lower_cols = {str(c).lower(): c for c in df.columns}
            ann_col = None
            val_col = None

            for k in ["annotationtype", "annotation_type", "annotation", "column", "field", "sdrf_column"]:
                if k in lower_cols:
                    ann_col = lower_cols[k]
                    break

            for k in ["value", "values", "prediction", "predicted_value", "answer", "text", "span"]:
                if k in lower_cols:
                    val_col = lower_cols[k]
                    break

            if ann_col is not None and val_col is not None:
                for _, r in df[[ann_col, val_col]].dropna().iterrows():
                    _append_value_pair(out, r[ann_col], r[val_col])

        else:
            raw = path.read_text(encoding="utf-8", errors="ignore")
            for line in raw.splitlines():
                m = re.match(r"^\s*(Characteristics\[[^\]]+\]|Comment\[[^\]]+\])\s*[:=-]\s*(.+?)\s*$", line, flags=re.I)
                if m:
                    _append_value_pair(out, m.group(1), m.group(2))

    except Exception:
        return {}

    return {
        k: list(dict.fromkeys(v))
        for k, v in out.items()
        if len(v) > 0
    }

TRAIN_PXDS = sorted([
    p.name.replace("_PubText.json", "")
    for p in TRAIN_PUB_DIR.glob("PXD*_PubText.json")
])

TEST_PXDS = sorted([
    p.name.replace("_PubText.json", "")
    for p in TEST_PUB_DIR.glob("PXD*_PubText.json")
])

print("Train PXDs:", len(TRAIN_PXDS))
print("Test PXDs :", len(TEST_PXDS))
print("Base metadata columns:", len(BASE_META_COLS))
print("GPT extract dir exists:", TRAIN_GPT_DIR.exists())

Train PXDs: 102
Test PXDs : 15
Base metadata columns: 72
GPT extract dir exists: True


In [25]:
# ================================
# CELL 2: BUILD TRAIN BANKS + NORMALIZATION MAPS + TEXT RETRIEVERS
# ================================

def normalize_for_match(x: str) -> str:
    x = unicodedata.normalize("NFKD", str(x))
    x = x.encode("ascii", "ignore").decode("ascii")
    x = x.lower()
    x = x.replace("β", "beta").replace("µ", "u")
    x = x.replace("_", " ")
    x = re.sub(r"[/\\|]+", " ", x)
    x = re.sub(r"[^a-z0-9+.%\- ]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def normalize_for_loose_text(x: str) -> str:
    x = normalize_for_match(x)
    x = re.sub(r"[-]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return f" {x} "

def is_real_value(x) -> bool:
    return normalize_for_match(str(x)) not in NA_VALUES

def unique_real_values_in_order(vals: Iterable) -> List[str]:
    out = []
    seen = set()
    for v in vals:
        s = clean_text(v)
        if not s or not is_real_value(s):
            continue
        k = normalize_for_match(s)
        if k not in seen:
            seen.add(k)
            out.append(s)
    return out

def get_base_values_from_df(df: pd.DataFrame, base_col: str) -> List[str]:
    cols = [c for c in df.columns if base_annotation(c) == base_col]
    vals = []
    for c in cols:
        vals.extend(df[c].tolist())
    return unique_real_values_in_order(vals)

TRAIN_RECORDS = []
COLUMN_NORM_TO_COUNTER = defaultdict(lambda: defaultdict(Counter))
COLUMN_PXD_VALUE_FREQ = defaultdict(Counter)
GPT_COVERAGE = Counter()

for pxd in TRAIN_PXDS:
    pub_json = load_pub_json(pxd, split="train")
    views = get_text_views(pub_json)
    gold = load_gold_sdrf(pxd)
    gpt_vals_raw = load_gpt_extract_as_value_dict(pxd)

    gold_values = {}
    gpt_values = {}
    union_values = {}

    for base_col in BASE_META_COLS:
        gvals = get_base_values_from_df(gold, base_col)
        xvals = unique_real_values_in_order(gpt_vals_raw.get(base_col, []))
        uvals = unique_real_values_in_order(gvals + xvals)

        gold_values[base_col] = gvals
        gpt_values[base_col] = xvals
        union_values[base_col] = uvals

        for v in gvals:
            COLUMN_NORM_TO_COUNTER[base_col][normalize_for_match(v)][v] += 3
            COLUMN_PXD_VALUE_FREQ[base_col][v] += 1

        for v in xvals:
            COLUMN_NORM_TO_COUNTER[base_col][normalize_for_match(v)][v] += 1

        if len(xvals) > 0:
            GPT_COVERAGE[base_col] += 1

    TRAIN_RECORDS.append({
        "pxd": pxd,
        "raw_files": extract_raw_files(pub_json),
        "raw_count": len(extract_raw_files(pub_json)),
        "title": views["title"],
        "abstract": views["abstract"],
        "methods_text": views["methods_text"],
        "sample_text": views["sample_text"],
        "results_text": views["results_text"],
        "caption_text": views["caption_text"],
        "core_text": views["core_text"],
        "full_text": views["full_text"],
        "gold_values": gold_values,
        "gpt_values": gpt_values,
        "union_values": union_values,
    })

TRAIN_BANK_DF = pd.DataFrame(TRAIN_RECORDS)

COLUMN_CANON_MAP = {}
COLUMN_VOCAB = {}

for base_col in BASE_META_COLS:
    canon_map = {}
    for norm_val, cnt in COLUMN_NORM_TO_COUNTER[base_col].items():
        canon_map[norm_val] = cnt.most_common(1)[0][0]

    vocab = []
    seen = set()

    for rec in TRAIN_RECORDS:
        for v in rec["union_values"][base_col]:
            vv = canon_map.get(normalize_for_match(v), v)
            k = normalize_for_match(vv)
            if k not in seen:
                seen.add(k)
                vocab.append(vv)

    COLUMN_CANON_MAP[base_col] = canon_map
    COLUMN_VOCAB[base_col] = vocab

def _safe_corpus(vals: List[str]) -> List[str]:
    if all(clean_text(v) == "" for v in vals):
        return ["blank"] * len(vals)
    return vals

FULL_CORPUS = _safe_corpus(TRAIN_BANK_DF["full_text"].fillna("").astype(str).tolist())
SAMPLE_CORPUS = _safe_corpus(TRAIN_BANK_DF["sample_text"].fillna("").astype(str).tolist())

STUDY_VECTORIZER_FULL = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=1,
    lowercase=True,
    strip_accents="unicode",
)

STUDY_VECTORIZER_SAMPLE = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=1,
    lowercase=True,
    strip_accents="unicode",
)

STUDY_X_FULL = STUDY_VECTORIZER_FULL.fit_transform(FULL_CORPUS)
STUDY_X_SAMPLE = STUDY_VECTORIZER_SAMPLE.fit_transform(SAMPLE_CORPUS)

print("TRAIN_BANK_DF shape:", TRAIN_BANK_DF.shape)
print("STUDY_X_FULL shape :", STUDY_X_FULL.shape)
print("STUDY_X_SAMPLE shape:", STUDY_X_SAMPLE.shape)

coverage_rows = []
for c in BASE_META_COLS:
    coverage_rows.append({
        "AnnotationType": c,
        "n_vocab": len(COLUMN_VOCAB[c]),
        "n_pxd_gold": sum(1 for r in TRAIN_RECORDS if len(r["gold_values"][c]) > 0),
        "n_pxd_gpt": GPT_COVERAGE[c],
    })

coverage_df = pd.DataFrame(coverage_rows).sort_values(
    ["n_pxd_gold", "n_vocab"], ascending=[False, False]
).reset_index(drop=True)

display(coverage_df.head(40))

TRAIN_BANK_DF shape: (102, 14)
STUDY_X_FULL shape : (102, 219456)
STUDY_X_SAMPLE shape: (102, 12)


,AnnotationType,n_vocab,n_pxd_gold,n_pxd_gpt
0,Comment[Instrument],332,102,0
1,Characteristics[CleavageAgent],222,102,0
2,Characteristics[Label],38,102,0
3,Characteristics[Organism],17,102,0
4,Usage,2,102,0
5,Comment[FractionIdentifier],114,99,0
6,Characteristics[BiologicalReplicate],41,97,0
7,Characteristics[Modification],1803,93,0
8,Characteristics[OrganismPart],112,84,0
9,Characteristics[Disease],117,77,0


In [26]:
# ================================
# CELL 3: EXPLICIT EXTRACTORS + FILENAME PARSING + CANDIDATE HELPERS (V5)
# ================================

import difflib

def _norm_alias_dict(d):
    return {normalize_for_match(k): v for k, v in d.items()}

VALUE_ALIAS_MAP = {
    "Characteristics[AlkylationReagent]": _norm_alias_dict({
        "IAA": "Iodoacetamide",
        "iodoacetamide": "Iodoacetamide",
        "chloroacetamide": "Chloroacetamide",
        "CAA": "Chloroacetamide",
        "NEM": "N-ethylmaleimide",
        "n-ethylmaleimide": "N-ethylmaleimide",
    }),
    "Characteristics[ReductionReagent]": _norm_alias_dict({
        "DTT": "DTT",
        "dithiothreitol": "DTT",
        "TCEP": "TCEP",
        "tris(2-carboxyethyl)phosphine": "TCEP",
        "beta-mercaptoethanol": "beta-mercaptoethanol",
        "mercaptoethanol": "beta-mercaptoethanol",
    }),
    "Comment[FragmentationMethod]": _norm_alias_dict({
        "higher-energy collisional dissociation": "HCD",
        "higher energy collisional dissociation": "HCD",
        "beam-type collision-induced dissociation": "HCD",
        "collision-induced dissociation": "CID",
        "collision induced dissociation": "CID",
        "electron transfer dissociation": "ETD",
        "electron capture dissociation": "ECD",
        "HCD": "HCD",
        "CID": "CID",
        "ETD": "ETD",
        "ECD": "ECD",
    }),
    "Comment[AcquisitionMethod]": _norm_alias_dict({
        "data dependent acquisition": "DDA",
        "data-dependent acquisition": "DDA",
        "data independent acquisition": "DIA",
        "data-independent acquisition": "DIA",
        "parallel reaction monitoring": "PRM",
        "selected reaction monitoring": "SRM",
        "multiple reaction monitoring": "MRM",
        "DDA": "DDA",
        "DIA": "DIA",
        "PRM": "PRM",
        "SRM": "SRM",
        "MRM": "MRM",
    }),
    "Comment[IonizationType]": _norm_alias_dict({
        "nanoelectrospray": "nanoESI",
        "nano electrospray": "nanoESI",
        "nanoesi": "nanoESI",
        "nanospray": "nanoESI",
        "esi": "ESI",
        "electrospray": "ESI",
        "maldi": "MALDI",
    }),
    "Comment[MS2MassAnalyzer]": _norm_alias_dict({
        "orbitrap": "orbitrap",
        "ion trap": "ion trap",
        "tof": "TOF",
        "time of flight": "TOF",
        "quadrupole": "quadrupole",
        "ft-icr": "FT-ICR",
        "ft icr": "FT-ICR",
    }),
    "Characteristics[Sex]": _norm_alias_dict({
        "male": "male",
        "males": "male",
        "man": "male",
        "men": "male",
        "female": "female",
        "females": "female",
        "woman": "female",
        "women": "female",
    }),
    "Characteristics[MaterialType]": _norm_alias_dict({
        "plasma": "biofluid",
        "serum": "biofluid",
        "urine": "biofluid",
        "saliva": "biofluid",
        "blood": "biofluid",
        "csf": "biofluid",
        "cerebrospinal fluid": "biofluid",
        "cell culture": "cell line",
        "cell line": "cell line",
        "cells": "primary cells",
        "tissue": "tissue",
        "biopsy": "tissue",
    }),
    "Characteristics[Label]": _norm_alias_dict({
        "label-free": "label free",
        "label free": "label free",
        "labelfree": "label free",
    }),
}

def normalize_age_value(x: str) -> str:
    s = clean_text(x).lower()
    s = s.replace("yrs", "years").replace("yr", "year").replace("mos", "months").replace("wks", "weeks")

    m = re.search(r"\b(\d{1,3})\s*-\s*year-old\b", s)
    if m:
        return f"{m.group(1)} years"

    m = re.search(r"\b(\d{1,3})\s*(years?|months?|weeks?|days?)\s*(?:old)?\b", s)
    if m:
        num = m.group(1)
        unit = m.group(2)
        unit = {
            "year": "years", "years": "years",
            "month": "months", "months": "months",
            "week": "weeks", "weeks": "weeks",
            "day": "days", "days": "days",
        }.get(unit, unit)
        return f"{num} {unit}"

    return clean_text(x)

def normalize_dev_stage_value(x: str) -> str:
    s = clean_text(x)
    if re.fullmatch(r"(?i)e\d+(?:\.\d+)?", s):
        return s.upper()
    if re.fullmatch(r"(?i)p\d+(?:\.\d+)?", s):
        return s.upper()

    low = s.lower()
    if low in {"adult", "embryonic", "fetal", "foetal", "newborn", "neonatal", "pup", "juvenile"}:
        return low
    return s

def normalize_collision_energy_value(x: str) -> str:
    s = clean_text(x)
    s = s.replace("NCE", "nce").replace("Ev", "eV").replace("EV", "eV")
    s = re.sub(r"\s+", " ", s)
    return s

def apply_alias_map(base_col: str, value: str) -> str:
    s = clean_text(value)
    norm = normalize_for_match(s)
    amap = VALUE_ALIAS_MAP.get(base_col, {})
    if norm in amap:
        return amap[norm]
    return s

def canonicalize_value(base_col: str, value: str, fuzzy_threshold: float = 0.92) -> Optional[str]:
    if value is None:
        return None

    s = clean_text(value)
    if not s or not is_real_value(s):
        return None

    s = apply_alias_map(base_col, s)

    if base_col == "Characteristics[Age]":
        s = normalize_age_value(s)
    elif base_col == "Characteristics[DevelopmentalStage]":
        s = normalize_dev_stage_value(s)
    elif base_col == "Comment[CollisionEnergy]":
        s = normalize_collision_energy_value(s)

    norm = normalize_for_match(s)

    if norm in COLUMN_CANON_MAP.get(base_col, {}):
        return COLUMN_CANON_MAP[base_col][norm]

    numeric_like_cols = {
        "Characteristics[Age]",
        "Characteristics[Time]",
        "Characteristics[SamplingTime]",
        "Characteristics[DevelopmentalStage]",
        "Comment[CollisionEnergy]",
        "Comment[PrecursorMassTolerance]",
        "Comment[FragmentMassTolerance]",
        "Comment[GradientTime]",
        "Comment[FlowRateChromatogram]",
        "Comment[NumberOfMissedCleavages]",
        "Comment[FractionIdentifier]",
    }

    if base_col in numeric_like_cols:
        return s

    vocab = COLUMN_VOCAB.get(base_col, [])
    if not vocab:
        return s

    best_val = None
    best_sim = -1.0

    for cand in vocab:
        sim = difflib.SequenceMatcher(None, norm, normalize_for_match(cand)).ratio()
        if sim > best_sim:
            best_sim = sim
            best_val = cand

    if best_val is not None and best_sim >= fuzzy_threshold:
        return best_val

    return s

def add_candidate(cands: Dict[str, Counter], base_col: str, value, score: float, source: str = ""):
    if value is None:
        return

    if isinstance(value, (list, tuple, set)):
        for v in value:
            add_candidate(cands, base_col, v, score, source)
        return

    can = canonicalize_value(base_col, value)
    if can is None:
        return

    cands[base_col][can] += float(score)

def select_best_value(cands: Dict[str, Counter], base_col: str, min_score: float = 1.0) -> Optional[str]:
    if base_col not in cands or len(cands[base_col]) == 0:
        return None
    val, sc = max(cands[base_col].items(), key=lambda kv: (kv[1], kv[0]))
    return val if sc >= min_score else None

def select_multi_values(
    cands: Dict[str, Counter],
    base_col: str,
    min_score: float = 1.0,
    max_values: int = 3,
) -> List[str]:
    if base_col not in cands or len(cands[base_col]) == 0:
        return []

    ranked = sorted(cands[base_col].items(), key=lambda kv: (-kv[1], kv[0]))
    out = []
    for v, sc in ranked:
        if sc >= min_score:
            out.append(v)
        if len(out) >= max_values:
            break
    return out

def _regex_span_matches(text: str, patterns: List[str]) -> List[str]:
    out = []
    seen = set()
    for pat in patterns:
        for m in re.finditer(pat, text, flags=re.I):
            s = clean_text(m.group(0))
            k = normalize_for_match(s)
            if s and k not in seen:
                seen.add(k)
                out.append(s)
    return out

def _regex_group1_matches(text: str, patterns: List[str]) -> List[str]:
    out = []
    seen = set()
    for pat in patterns:
        for m in re.finditer(pat, text, flags=re.I):
            s = clean_text(m.group(1) if m.lastindex else m.group(0))
            k = normalize_for_match(s)
            if s and k not in seen:
                seen.add(k)
                out.append(s)
    return out

def split_sentences(text: str) -> List[str]:
    text = clean_text(text)
    if not text:
        return []
    parts = re.split(r'(?<=[\.\?\!;])\s+|\n+', text)
    parts = [clean_text(p) for p in parts]
    return [p for p in parts if len(p.split()) >= 4]

def get_anchor_windows(text: str, anchor_patterns: List[str], radius: int = 220) -> List[str]:
    text = clean_text(text)
    if not text:
        return []

    wins = []
    for pat in anchor_patterns:
        for m in re.finditer(pat, text, flags=re.I):
            st = max(0, m.start() - radius)
            en = min(len(text), m.end() + radius)
            wins.append(text[st:en])

    return _unique_chunks(wins)

NOISY_GENERIC_VALUES = {
    "male", "female", "adult", "control", "treated", "untreated",
    "healthy", "normal", "tumor", "tumour", "disease"
}

def vocab_mentions_in_text(base_col: str, text: str, max_hits: int = 12) -> List[str]:
    loose = normalize_for_loose_text(text)
    vocab = sorted(COLUMN_VOCAB.get(base_col, []), key=lambda x: (-len(normalize_for_match(x)), x))

    hits = []
    seen = set()

    for v in vocab:
        nv = normalize_for_match(v)
        if not nv:
            continue
        if nv in NOISY_GENERIC_VALUES:
            continue
        if len(nv) < 4 and not re.search(r"\d", nv):
            continue

        if f" {nv} " in loose:
            if nv not in seen:
                seen.add(nv)
                hits.append(v)
        if len(hits) >= max_hits:
            break

    return hits

def ranked_vocab_mentions(base_col: str, texts: List[str], max_values: int = 8) -> List[Tuple[str, int]]:
    ctr = Counter()
    for txt in texts:
        for v in vocab_mentions_in_text(base_col, txt, max_hits=200):
            vv = canonicalize_value(base_col, v)
            if vv is not None:
                ctr[vv] += 1
    return ctr.most_common(max_values)

INSTRUMENT_PATTERNS = [
    r"\bq[- ]?exactive(?: plus| hf(?:-x)?| classic)?\b",
    r"\bqe[- ]?hf(?:-x)?\b",
    r"\borbitrap fusion(?: lumos)?\b",
    r"\bfusion lumos\b",
    r"\bltq[- ]?orbitrap(?: velos| elite| xl)?\b",
    r"\borbitrap elite\b",
    r"\borbitrap velos(?: pro)?\b",
    r"\bvelos pro\b",
    r"\btriple[- ]?tof(?: 5600| 6600)?\b",
    r"\btims[- ]?tof(?: pro)?\b",
    r"\bq[- ]?tof\b",
    r"\bmaldi[- ]?tof\b",
]

LABEL_PATTERNS = [
    r"\blabel[- ]?free(?: sample)?\b",
    r"\btmtpro[- ]?\d+[nc]?\b",
    r"\btmt[- ]?\d+[nc]?\b",
    r"\bitraq[- ]?\d+\b",
    r"\bsilac(?:[- ]?(?:heavy|light|medium))?\b",
]

FRAGMENTATION_PATTERNS = [r"\bhcd\b", r"\bcid\b", r"\betd\b", r"\becd\b", r"\bpqd\b"]
ACQUISITION_PATTERNS = [r"\bdda\b", r"\bdia\b", r"\bprm\b", r"\bsrm\b", r"\bmrm\b"]
CLEAVAGE_PATTERNS = [
    r"\btrypsin\b", r"\blys[- ]?c\b", r"\bchymotrypsin\b",
    r"\bglu[- ]?c\b", r"\basp[- ]?n\b", r"\barg[- ]?c\b", r"\bpepsin\b"
]
ENRICHMENT_PATTERNS = [
    r"\bimac\b", r"\btio2\b", r"\btitanium dioxide\b",
    r"\banti-?phosphotyrosine\b", r"\bscx\b", r"\bfe[- ]?nimac\b"
]
IONIZATION_PATTERNS = [r"\bnanoesi\b", r"\besi\b", r"\bmaldi\b"]
ANALYZER_PATTERNS = [r"\borbitrap\b", r"\bion trap\b", r"\bquadrupole\b", r"\btof\b", r"\bft[- ]?icr\b"]
ALKYLATION_PATTERNS = [r"\biodoacetamide\b", r"\biaa\b", r"\bchloroacetamide\b", r"\bcaa\b", r"\bnem\b"]
REDUCTION_PATTERNS = [r"\bdtt\b", r"\btcep\b", r"\bbeta[- ]?mercaptoethanol\b", r"\bmercaptoethanol\b"]
MODIFICATION_PATTERNS = [
    r"\bphosphorylat(?:ion|ed)\b",
    r"\bphosphoprote(?:ome|omics?)\b",
    r"\bphosphopeptide(?:s)?\b",
    r"\bubiquitin(?:ation|ylation)?\b",
    r"\bacetyl(?:ation|ome)?\b",
    r"\bglycosylat(?:ion|ed)\b",
    r"\bmethylat(?:ion|ed)\b",
    r"\bsuccinylat(?:ion|ed)\b",
    r"\bcrotonylat(?:ion|ed)\b",
]

AGE_PATTERNS = [
    r"\b(\d{1,3}\s*(?:years?|yrs?|months?|mos?|weeks?|wks?|days?)\s*(?:old)?)\b",
    r"\b(\d{1,3}-year-old)\b",
]
DEV_STAGE_PATTERNS = [
    r"\b(E\d+(?:\.\d+)?)\b",
    r"\b(P\d+(?:\.\d+)?)\b",
    r"\b(adult|embryonic|fetal|foetal|newborn|neonatal|pup|juvenile)\b",
]
FLOW_RATE_PATTERNS = [
    r"(?:flow rate(?: of)?\s*)(\d+(?:\.\d+)?\s?(?:nl/min|ul/min|µl/min|ml/min))",
    r"(\d+(?:\.\d+)?\s?(?:nl/min|ul/min|µl/min|ml/min)\s+flow rate)",
]
GRADIENT_TIME_PATTERNS = [
    r"(?:gradient(?: length| time)?(?: of)?\s*)(\d+(?:\.\d+)?\s?(?:min|minutes))",
    r"(\d+(?:\.\d+)?\s?(?:min|minutes)\s+gradient)",
]
PRECURSOR_TOL_PATTERNS = [
    r"(?:precursor(?: ion| mass)? tolerance(?: of)?\s*)(\d+(?:\.\d+)?\s?(?:ppm|da))",
    r"(\d+(?:\.\d+)?\s?(?:ppm|da)\s+precursor(?: ion| mass)? tolerance)",
]
FRAGMENT_TOL_PATTERNS = [
    r"(?:fragment(?: ion| mass)? tolerance(?: of)?\s*)(\d+(?:\.\d+)?\s?(?:ppm|da))",
    r"(\d+(?:\.\d+)?\s?(?:ppm|da)\s+fragment(?: ion| mass)? tolerance)",
]
MISSED_CLEAVAGES_PATTERNS = [
    r"(?:up to |maximum of |max(?:imum)? )?(\d+)\s+missed cleavages?\b",
]
COLLISION_ENERGY_PATTERNS = [
    r"(?:collision energy(?: of)?\s*)(\d+(?:\.\d+)?\s?(?:ev|eV|%|nce))",
    r"(?:normalized collision energy(?: of)?\s*)(\d+(?:\.\d+)?\s?(?:%|nce)?)",
    r"\bNCE\s*[:=]?\s*(\d+(?:\.\d+)?)\b",
    r"\bCE\s*[:=]?\s*(\d+(?:\.\d+)?)\s*(?:eV|%)?\b",
]

FRACTIONATION_PATTERNS = [
    r"\bhigh[- ]?pH reverse[- ]?phase\b",
    r"\boff-?gel\b",
    r"\bstrong cation exchange\b",
    r"\bscx\b",
    r"\bsds[- ]?page\b",
    r"\bGELFrEE\b",
    r"\bhigh[- ]?resolution isoelectric focusing\b",
]
SEPARATION_PATTERNS = [
    r"\bnano[- ]?lc\b",
    r"\bcapillary lc\b",
    r"\breverse[- ]?phase (?:liquid chromatography|lc)\b",
    r"\buhplc\b",
    r"\b2d[- ]?lc\b",
]

DISEASE_ANCHORS = [
    r"\bpatient[s]?\b", r"\bcohort\b", r"\bdisease\b", r"\bhealthy\b",
    r"\bcontrol[s]?\b", r"\bcancer\b", r"\bcarcinoma\b", r"\btumou?r\b",
    r"\bdiagnos", r"\bcase[s]?\b"
]
ORGPART_ANCHORS = [
    r"\btissue\b", r"\bbiops", r"\bspecimen\b", r"\borgan\b",
    r"\bplasma\b", r"\bserum\b", r"\burine\b", r"\bblood\b",
    r"\btumou?r\b", r"\blesion\b"
]
SAMPLE_ANCHORS = [
    r"\bsample[s]?\b", r"\bsubject[s]?\b", r"\bpatient[s]?\b",
    r"\bcell(?:s)?\b", r"\btissue\b", r"\bbiops", r"\bplasma\b", r"\bserum\b"
]
SEX_ANCHORS = [r"\bmale\b", r"\bfemale\b", r"\bmen\b", r"\bwomen\b"]
AGE_ANCHORS = [r"\bage\b", r"\byear-old\b", r"\bmonths?\b", r"\bweeks?\b", r"\bdays?\b"]
DEV_ANCHORS = [r"\bembry", r"\bfetal\b", r"\bfoetal\b", r"\bneonat", r"\bnewborn\b", r"\badult\b", r"\bE\d+", r"\bP\d+"]

def parse_filename_metadata_per_file(raw_file: str) -> Dict[str, str]:
    name = Path(str(raw_file)).name
    stem = name.rsplit(".", 1)[0]
    out = {}

    frac_patterns = [
        r"(?i)(?:^|[_\-.])(f|fr|fx|frac|fraction|band|slice|pool|offgel|scx|hph)[-_ ]*0*([0-9]{1,3}|[a-h])(?:$|[_\-.])",
        r"(?i)(?:^|[_\-.])f0*([0-9]{1,3})(?:$|[_\-.])",
    ]
    for pat in frac_patterns:
        m = re.search(pat, stem)
        if m:
            out["Comment[FractionIdentifier]"] = m.group(m.lastindex)
            break

    rep_patterns = [
        r"(?i)(?:^|[_\-.])(?:biorep|bio(?:logical)?rep(?:licate)?|replicate|rep)[-_ ]*0*([1-9][0-9]?)(?:$|[_\-.])",
        r"(?i)(?:^|[_\-.])r0*([1-9][0-9]?)(?:$|[_\-.])",
    ]
    for pat in rep_patterns:
        m = re.search(pat, stem)
        if m:
            out["Characteristics[BiologicalReplicate]"] = m.group(1)
            break

    m = re.search(r"(?i)(tmtpro[- ]?\d+[nc]?|tmt[- ]?\d+[nc]?|itraq[- ]?\d+|silac(?:[-_ ]?(?:heavy|light|medium))?)", stem)
    if m:
        out["Characteristics[Label]"] = m.group(1)

    m = re.search(r"(?i)(?:^|[_\-.])(\d+(?:h|hr|hrs|hour|hours|d|day|days|w|week|weeks))(?:$|[_\-.])", stem)
    if m:
        out["Characteristics[Time]"] = m.group(1)

    if re.search(r"(?i)(?:^|[_\-.])(ctrl|control|vehicle|untreated)(?:$|[_\-.])", stem):
        out["Characteristics[Treatment]"] = "control"
    elif re.search(r"(?i)(?:^|[_\-.])(treated|drug|stim|stimulated)(?:$|[_\-.])", stem):
        out["Characteristics[Treatment]"] = "treated"

    if re.search(r"(?i)(?:^|[_\-.])(ko|knockout)(?:$|[_\-.])", stem):
        out["Characteristics[GeneticModification]"] = "knockout"
    elif re.search(r"(?i)(?:^|[_\-.])(wt|wildtype|wild-type)(?:$|[_\-.])", stem):
        out["Characteristics[Genotype]"] = "wild type"

    return {
        k: canonicalize_value(k, v) if k in BASE_META_COLS else v
        for k, v in out.items()
    }

def infer_study_hints_from_filenames(raw_files: List[str]) -> Dict[str, List[str]]:
    counts = defaultdict(Counter)
    for rf in raw_files:
        md = parse_filename_metadata_per_file(rf)
        for k, v in md.items():
            if v is not None:
                counts[k][v] += 1

    out = {}
    for k, ctr in counts.items():
        out[k] = [v for v, _ in sorted(ctr.items(), key=lambda kv: (-kv[1], kv[0]))]
    return out

def collect_explicit_candidates(pub_json: dict, raw_files: List[str]):
    views = get_text_views(pub_json)

    full_text = views["full_text"]
    methods_text = views["methods_text"] or views["core_text"]
    sampleish_text = " ".join([
        views["title"], views["abstract"], views["sample_text"],
        views["results_text"], views["caption_text"]
    ])

    cands = defaultdict(Counter)

    # Strong protocol regexes
    for v in _regex_span_matches(methods_text + " " + full_text, INSTRUMENT_PATTERNS):
        add_candidate(cands, "Comment[Instrument]", v, 5.0, "instrument_regex")

    for v in _regex_span_matches(full_text, LABEL_PATTERNS):
        add_candidate(cands, "Characteristics[Label]", v, 4.0, "label_regex")

    for v in _regex_span_matches(methods_text + " " + full_text, FRAGMENTATION_PATTERNS):
        add_candidate(cands, "Comment[FragmentationMethod]", v, 4.0, "frag_regex")

    for v in _regex_span_matches(methods_text + " " + full_text, ACQUISITION_PATTERNS):
        add_candidate(cands, "Comment[AcquisitionMethod]", v, 4.0, "acq_regex")

    for v in _regex_span_matches(methods_text + " " + full_text, CLEAVAGE_PATTERNS):
        add_candidate(cands, "Characteristics[CleavageAgent]", v, 4.0, "cleavage_regex")

    for v in _regex_span_matches(methods_text + " " + full_text, ENRICHMENT_PATTERNS):
        add_candidate(cands, "Comment[EnrichmentMethod]", v, 3.8, "enrich_regex")

    for v in _regex_span_matches(methods_text + " " + full_text, IONIZATION_PATTERNS):
        add_candidate(cands, "Comment[IonizationType]", v, 3.8, "ion_regex")

    for v in _regex_span_matches(methods_text + " " + full_text, ANALYZER_PATTERNS):
        add_candidate(cands, "Comment[MS2MassAnalyzer]", v, 3.8, "analyzer_regex")

    for v in _regex_span_matches(methods_text + " " + full_text, ALKYLATION_PATTERNS):
        add_candidate(cands, "Characteristics[AlkylationReagent]", v, 4.0, "alkyl_regex")

    for v in _regex_span_matches(methods_text + " " + full_text, REDUCTION_PATTERNS):
        add_candidate(cands, "Characteristics[ReductionReagent]", v, 4.0, "reduction_regex")

    for v in _regex_span_matches(methods_text + " " + full_text, MODIFICATION_PATTERNS):
        add_candidate(cands, "Characteristics[Modification]", v, 3.2, "mod_regex")

    for v in _regex_span_matches(methods_text + " " + full_text, FRACTIONATION_PATTERNS):
        add_candidate(cands, "Comment[FractionationMethod]", v, 3.2, "fractionation_regex")

    for v in _regex_span_matches(methods_text + " " + full_text, SEPARATION_PATTERNS):
        add_candidate(cands, "Comment[Separation]", v, 3.0, "separation_regex")

    # Numeric-ish protocol fields
    for v in _regex_group1_matches(methods_text + " " + full_text, FLOW_RATE_PATTERNS):
        add_candidate(cands, "Comment[FlowRateChromatogram]", v, 3.0, "flow_regex")

    for v in _regex_group1_matches(methods_text + " " + full_text, GRADIENT_TIME_PATTERNS):
        add_candidate(cands, "Comment[GradientTime]", v, 3.0, "grad_regex")

    for v in _regex_group1_matches(methods_text + " " + full_text, PRECURSOR_TOL_PATTERNS):
        add_candidate(cands, "Comment[PrecursorMassTolerance]", v, 3.0, "prec_tol_regex")

    for v in _regex_group1_matches(methods_text + " " + full_text, FRAGMENT_TOL_PATTERNS):
        add_candidate(cands, "Comment[FragmentMassTolerance]", v, 3.0, "frag_tol_regex")

    for v in _regex_group1_matches(methods_text + " " + full_text, MISSED_CLEAVAGES_PATTERNS):
        add_candidate(cands, "Comment[NumberOfMissedCleavages]", v, 3.0, "missed_regex")

    for v in _regex_group1_matches(methods_text + " " + full_text, COLLISION_ENERGY_PATTERNS):
        add_candidate(cands, "Comment[CollisionEnergy]", v, 3.0, "collision_regex")

    # Direct vocab scans for protocol columns
    for base_col, text, score in [
        ("Characteristics[Organism]", full_text, 3.2),
        ("Characteristics[CellLine]", sampleish_text, 3.0),
        ("Characteristics[CellType]", sampleish_text, 2.8),
        ("Characteristics[Modification]", methods_text + " " + full_text, 2.8),
        ("Comment[Instrument]", methods_text + " " + full_text, 3.4),
        ("Comment[EnrichmentMethod]", methods_text + " " + full_text, 2.8),
        ("Characteristics[CleavageAgent]", methods_text + " " + full_text, 2.8),
        ("Comment[FractionationMethod]", methods_text + " " + full_text, 2.6),
        ("Comment[Separation]", methods_text + " " + full_text, 2.6),
    ]:
        for v in vocab_mentions_in_text(base_col, text):
            add_candidate(cands, base_col, v, score, "vocab_scan")

    # Context-aware sample windows for harder columns
    disease_windows = get_anchor_windows(sampleish_text, DISEASE_ANCHORS, radius=220)
    orgpart_windows = get_anchor_windows(sampleish_text, ORGPART_ANCHORS, radius=220)
    sex_windows = get_anchor_windows(sampleish_text, SEX_ANCHORS, radius=140)
    age_windows = get_anchor_windows(sampleish_text, AGE_ANCHORS, radius=140)
    dev_windows = get_anchor_windows(sampleish_text, DEV_ANCHORS, radius=160)
    sample_windows = get_anchor_windows(sampleish_text, SAMPLE_ANCHORS, radius=220)

    if not disease_windows:
        disease_windows = split_sentences(sampleish_text)[:25]
    if not orgpart_windows:
        orgpart_windows = split_sentences(sampleish_text)[:25]

    for v, cnt in ranked_vocab_mentions("Characteristics[Disease]", disease_windows, max_values=6):
        add_candidate(cands, "Characteristics[Disease]", v, 2.2 + 0.5 * cnt, "disease_window_vocab")

    for v, cnt in ranked_vocab_mentions("Characteristics[OrganismPart]", orgpart_windows + sample_windows, max_values=8):
        add_candidate(cands, "Characteristics[OrganismPart]", v, 2.2 + 0.45 * cnt, "orgpart_window_vocab")

    for v, cnt in ranked_vocab_mentions("Characteristics[CellType]", sample_windows, max_values=6):
        add_candidate(cands, "Characteristics[CellType]", v, 1.8 + 0.35 * cnt, "celltype_window_vocab")

    for v, cnt in ranked_vocab_mentions("Characteristics[CellLine]", sample_windows, max_values=4):
        add_candidate(cands, "Characteristics[CellLine]", v, 2.0 + 0.4 * cnt, "cellline_window_vocab")

    # MaterialType rules
    loose_sample = normalize_for_loose_text(sampleish_text)

    if any(x in loose_sample for x in [" plasma ", " serum ", " urine ", " saliva ", " cerebrospinal fluid ", " csf ", " blood "]):
        add_candidate(cands, "Characteristics[MaterialType]", "biofluid", 4.0, "mat_rule")

    if any(x in loose_sample for x in [" cell line ", " hela ", " hek293 ", " hek293t ", " u2os ", " a549 ", " k562 ", " jurkat "]):
        add_candidate(cands, "Characteristics[MaterialType]", "cell line", 4.0, "mat_rule")

    if any(x in loose_sample for x in [" tissue ", " biopsy ", " tumor ", " tumour ", " resection ", " organ "]):
        add_candidate(cands, "Characteristics[MaterialType]", "tissue", 3.8, "mat_rule")

    # Sex, allow both
    if re.search(r"\b(male|males|men|man)\b", sampleish_text, flags=re.I):
        add_candidate(cands, "Characteristics[Sex]", "male", 4.0, "sex_rule")
    if re.search(r"\b(female|females|women|woman)\b", sampleish_text, flags=re.I):
        add_candidate(cands, "Characteristics[Sex]", "female", 4.0, "sex_rule")

    # Age
    for v in _regex_group1_matches(" ".join(age_windows) + " " + sampleish_text, AGE_PATTERNS):
        add_candidate(cands, "Characteristics[Age]", v, 3.2, "age_regex")

    # Developmental stage
    for v in _regex_group1_matches(" ".join(dev_windows) + " " + sampleish_text, DEV_STAGE_PATTERNS):
        add_candidate(cands, "Characteristics[DevelopmentalStage]", v, 3.2, "dev_stage_regex")

    # Filename-derived study hints
    file_hints = infer_study_hints_from_filenames(raw_files)

    for v in file_hints.get("Characteristics[Label]", []):
        add_candidate(cands, "Characteristics[Label]", v, 3.6, "filename_study")

    for v in file_hints.get("Characteristics[BiologicalReplicate]", []):
        add_candidate(cands, "Characteristics[BiologicalReplicate]", v, 3.2, "filename_study")

    for v in file_hints.get("Comment[FractionIdentifier]", []):
        add_candidate(cands, "Comment[FractionIdentifier]", v, 3.2, "filename_study")

    for v in file_hints.get("Characteristics[Treatment]", []):
        add_candidate(cands, "Characteristics[Treatment]", v, 2.6, "filename_study")

    for v in file_hints.get("Characteristics[Time]", []):
        add_candidate(cands, "Characteristics[Time]", v, 2.2, "filename_study")

    return cands, views, file_hints

In [27]:
# ================================
# CELL 4: RETRIEVAL + EVIDENCE FUSION (V5)
# ================================

MULTI_VALUE_COLS = {
    "Characteristics[Label]": {"max_values": 4, "min_score": 1.8},
    "Characteristics[BiologicalReplicate]": {"max_values": 12, "min_score": 1.8},
    "Characteristics[Modification]": {"max_values": 8, "min_score": 1.6},
    "Characteristics[OrganismPart]": {"max_values": 4, "min_score": 1.6},
    "Characteristics[Disease]": {"max_values": 3, "min_score": 1.6},
    "Characteristics[Sex]": {"max_values": 2, "min_score": 1.4},
    "Characteristics[DevelopmentalStage]": {"max_values": 3, "min_score": 1.5},
}

SINGLE_VALUE_MIN_SCORE = {
    "Characteristics[Organism]": 1.8,
    "Characteristics[CellType]": 1.8,
    "Characteristics[CellLine]": 1.8,
    "Characteristics[MaterialType]": 1.6,
    "Characteristics[Age]": 1.5,
    "Characteristics[Treatment]": 1.5,
    "Characteristics[Time]": 1.5,
    "Characteristics[AlkylationReagent]": 1.6,
    "Characteristics[ReductionReagent]": 1.6,
    "Characteristics[CleavageAgent]": 1.8,
    "Comment[Instrument]": 2.0,
    "Comment[FragmentationMethod]": 1.8,
    "Comment[AcquisitionMethod]": 1.8,
    "Comment[EnrichmentMethod]": 1.6,
    "Comment[IonizationType]": 1.6,
    "Comment[MS2MassAnalyzer]": 1.6,
    "Comment[CollisionEnergy]": 1.4,
    "Comment[PrecursorMassTolerance]": 1.6,
    "Comment[FragmentMassTolerance]": 1.6,
    "Comment[FlowRateChromatogram]": 1.6,
    "Comment[GradientTime]": 1.6,
    "Comment[NumberOfMissedCleavages]": 1.6,
    "Comment[FractionationMethod]": 1.5,
    "Comment[Separation]": 1.5,
}

RETRIEVAL_SINGLE_CONFIG = {
    "Characteristics[Organism]": {"min_share": 0.34},
    "Characteristics[CellType]": {"min_share": 0.24},
    "Characteristics[CellLine]": {"min_share": 0.24},
    "Characteristics[MaterialType]": {"min_share": 0.22},
    "Characteristics[CleavageAgent]": {"min_share": 0.28},
    "Comment[Instrument]": {"min_share": 0.24},
    "Comment[FragmentationMethod]": {"min_share": 0.24},
    "Comment[AcquisitionMethod]": {"min_share": 0.24},
    "Comment[EnrichmentMethod]": {"min_share": 0.20},
    "Comment[IonizationType]": {"min_share": 0.20},
    "Comment[MS2MassAnalyzer]": {"min_share": 0.20},
    "Characteristics[AlkylationReagent]": {"min_share": 0.20},
    "Characteristics[ReductionReagent]": {"min_share": 0.20},
    "Comment[PrecursorMassTolerance]": {"min_share": 0.20},
    "Comment[FragmentMassTolerance]": {"min_share": 0.20},
    "Comment[FlowRateChromatogram]": {"min_share": 0.20},
    "Comment[GradientTime]": {"min_share": 0.20},
    "Comment[NumberOfMissedCleavages]": {"min_share": 0.20},
    "Comment[FractionationMethod]": {"min_share": 0.18},
    "Comment[Separation]": {"min_share": 0.18},
}

RETRIEVAL_MULTI_CONFIG = {
    "Characteristics[Label]": {"min_share": 0.16, "max_values": 4},
    "Characteristics[BiologicalReplicate]": {"min_share": 0.12, "max_values": 12},
    "Characteristics[Modification]": {"min_share": 0.14, "max_values": 8},
    "Characteristics[OrganismPart]": {"min_share": 0.12, "max_values": 4},
    "Characteristics[Disease]": {"min_share": 0.12, "max_values": 3},
    "Characteristics[Sex]": {"min_share": 0.10, "max_values": 2},
    "Characteristics[DevelopmentalStage]": {"min_share": 0.10, "max_values": 3},
}

def get_neighbors_for_pub(
    pub_json: dict,
    exclude_pxd: Optional[str] = None,
    top_k: int = 12,
    min_sim: float = 0.04,
) -> pd.DataFrame:
    views = get_text_views(pub_json)

    q_full = clean_text(views["full_text"])
    q_sample = clean_text(views["sample_text"] or views["core_text"])

    qX_full = STUDY_VECTORIZER_FULL.transform([q_full if q_full else "blank"])
    qX_sample = STUDY_VECTORIZER_SAMPLE.transform([q_sample if q_sample else "blank"])

    sims_full = linear_kernel(qX_full, STUDY_X_FULL).ravel()
    sims_sample = linear_kernel(qX_sample, STUDY_X_SAMPLE).ravel()

    sims = 0.68 * sims_full + 0.32 * sims_sample
    order = np.argsort(-sims)

    rows = []
    for idx in order:
        pxd = TRAIN_BANK_DF.iloc[idx]["pxd"]
        sim = float(sims[idx])

        if exclude_pxd is not None and pxd == exclude_pxd:
            continue
        if sim < min_sim:
            continue

        rows.append({"idx": int(idx), "pxd": pxd, "sim": sim})
        if len(rows) >= top_k:
            break

    return pd.DataFrame(rows)

def _neighbor_values_for_col(rec: dict, base_col: str) -> List[str]:
    gold_vals = rec["gold_values"].get(base_col, [])
    if gold_vals:
        return gold_vals
    return rec["gpt_values"].get(base_col, [])

def neighbor_ranked_values(neighbors_df: pd.DataFrame, base_col: str) -> List[Tuple[str, float, float, int]]:
    if len(neighbors_df) == 0:
        return []

    weights = Counter()
    support_weight = 0.0
    hit_count = Counter()

    for _, nr in neighbors_df.iterrows():
        idx = int(nr["idx"])
        sim = float(nr["sim"])
        rec = TRAIN_BANK_DF.iloc[idx]
        vals = _neighbor_values_for_col(rec, base_col)

        if not vals:
            continue

        support_weight += sim
        for v in vals:
            vv = canonicalize_value(base_col, v)
            if vv is None:
                continue
            weights[vv] += sim
            hit_count[vv] += 1

    if support_weight <= 0 or len(weights) == 0:
        return []

    ranked = []
    for v, w in weights.items():
        share = w / support_weight
        ranked.append((v, share, w, hit_count[v]))

    ranked.sort(key=lambda x: (-x[1], -x[3], x[0]))
    return ranked

def add_neighbor_candidates(pub_json: dict, cands: Dict[str, Counter], exclude_pxd: Optional[str] = None):
    neighbors_df = get_neighbors_for_pub(pub_json, exclude_pxd=exclude_pxd, top_k=12, min_sim=0.04)

    for base_col, cfg in RETRIEVAL_SINGLE_CONFIG.items():
        ranked = neighbor_ranked_values(neighbors_df, base_col)
        if not ranked:
            continue

        best_val, best_share, _, best_hits = ranked[0]
        if best_share >= cfg["min_share"]:
            score = 0.85 + 2.15 * best_share + 0.12 * min(best_hits, 3)
            add_candidate(cands, base_col, best_val, score, "neighbor_single")

    for base_col, cfg in RETRIEVAL_MULTI_CONFIG.items():
        ranked = neighbor_ranked_values(neighbors_df, base_col)
        if not ranked:
            continue

        for val, share, _, hits in ranked[:cfg["max_values"]]:
            if share >= cfg["min_share"]:
                score = 0.70 + 1.85 * share + 0.10 * min(hits, 3)
                add_candidate(cands, base_col, val, score, "neighbor_multi")

    return neighbors_df

def build_prediction_evidence(pub_json: dict, raw_files: List[str], exclude_pxd: Optional[str] = None):
    cands, views, file_hints = collect_explicit_candidates(pub_json, raw_files)
    neighbors_df = add_neighbor_candidates(pub_json, cands, exclude_pxd=exclude_pxd)

    single_preds = {}
    multi_preds = {}

    for base_col in BASE_META_COLS:
        if base_col in MULTI_VALUE_COLS:
            continue
        if base_col == "Comment[FractionIdentifier]":
            continue
        if base_col == "Characteristics[BiologicalReplicate]":
            continue

        min_sc = SINGLE_VALUE_MIN_SCORE.get(base_col, 1.6)
        val = select_best_value(cands, base_col, min_score=min_sc)
        if val is not None:
            single_preds[base_col] = val

    for base_col, cfg in MULTI_VALUE_COLS.items():
        vals = select_multi_values(
            cands,
            base_col,
            min_score=cfg["min_score"],
            max_values=cfg["max_values"],
        )
        if vals:
            multi_preds[base_col] = vals

    if "Characteristics[MaterialType]" not in single_preds:
        loose = normalize_for_loose_text(views["sample_text"] + " " + views["full_text"])
        if any(x in loose for x in [" plasma ", " serum ", " urine ", " saliva ", " blood "]):
            single_preds["Characteristics[MaterialType]"] = canonicalize_value("Characteristics[MaterialType]", "biofluid")
        elif "Characteristics[CellLine]" in single_preds:
            single_preds["Characteristics[MaterialType]"] = canonicalize_value("Characteristics[MaterialType]", "cell line")
        elif "Characteristics[OrganismPart]" in multi_preds or "Characteristics[OrganismPart]" in single_preds:
            single_preds["Characteristics[MaterialType]"] = canonicalize_value("Characteristics[MaterialType]", "tissue")

    return single_preds, multi_preds, cands, neighbors_df, file_hints, views

In [28]:
# ================================
# CELL 5: V5 PREDICTOR
# ================================

def get_base_value_from_row(row: dict, base_col: str) -> Optional[str]:
    cols = BASE_TO_SUBCOLS.get(base_col, [])
    if not cols:
        return None
    return row.get(cols[0], DEFAULT_FILL)

def set_base_value(row: dict, base_col: str, value: str, overwrite: bool = False):
    cols = BASE_TO_SUBCOLS.get(base_col, [])
    if not cols or value is None:
        return

    col = cols[0]
    current = row.get(col, DEFAULT_FILL)
    if overwrite or current == DEFAULT_FILL:
        row[col] = str(value)

def set_base_values_across_slots(row: dict, base_col: str, values: List[str]):
    cols = BASE_TO_SUBCOLS.get(base_col, [])
    if not cols:
        return

    for c in cols:
        row[c] = DEFAULT_FILL

    for c, v in zip(cols, values):
        row[c] = str(v)

def assign_round_robin_base(rows: List[dict], base_col: str, values: List[str], overwrite_if_default_only: bool = True):
    if len(rows) == 0 or len(values) == 0:
        return

    cols = BASE_TO_SUBCOLS.get(base_col, [])
    if not cols:
        return

    col = cols[0]
    n = len(values)

    for i, row in enumerate(rows):
        if overwrite_if_default_only and row.get(col, DEFAULT_FILL) != DEFAULT_FILL:
            continue
        row[col] = str(values[i % n])

def fit_prediction_to_target_rows(pred_df: pd.DataFrame, target_n: int) -> pd.DataFrame:
    pred_df = pred_df.copy().reset_index(drop=True)

    if len(pred_df) == 0:
        out = pd.DataFrame([{col: DEFAULT_FILL for col in SUB_COLUMNS} for _ in range(target_n)])
        return out[SUB_COLUMNS]

    if len(pred_df) == target_n:
        return pred_df[SUB_COLUMNS].copy()

    if len(pred_df) < target_n:
        reps = int(np.ceil(target_n / len(pred_df)))
        out = pd.concat([pred_df] * reps, ignore_index=True).iloc[:target_n].copy()
        return out[SUB_COLUMNS]

    out = pred_df.iloc[:target_n].copy().reset_index(drop=True)
    return out[SUB_COLUMNS]

def _rep_sort_key(x):
    m = re.search(r"\d+", str(x))
    return int(m.group(0)) if m else 10**9

STATIC_MULTI_SLOT_COLS = {
    "Characteristics[Modification]",
    "Characteristics[OrganismPart]",
    "Characteristics[Disease]",
    "Characteristics[Sex]",
    "Characteristics[DevelopmentalStage]",
}

def baseline_predict_pxd_v5(pxd: str, split: str = "train", debug: bool = False) -> pd.DataFrame:
    pub_json = load_pub_json(pxd, split=split)
    raw_files = extract_raw_files(pub_json)

    if len(raw_files) == 0:
        raw_files = [f"{pxd}_file1.raw"]

    exclude = pxd if split == "train" else None

    single_preds, multi_preds, cands, neighbors_df, file_hints, views = build_prediction_evidence(
        pub_json,
        raw_files,
        exclude_pxd=exclude,
    )

    rows = []
    for i, raw_file in enumerate(raw_files, start=1):
        row = make_blank_row()

        if "ID" in row:
            row["ID"] = i
        if "PXD" in row:
            row["PXD"] = pxd
        if "Raw Data File" in row:
            row["Raw Data File"] = raw_file
        if "Source Name" in row:
            row["Source Name"] = f"{pxd}_source_{i}"
        if "Assay Name" in row:
            row["Assay Name"] = Path(str(raw_file)).stem[:200]

        # Global single-value metadata
        for base_col, val in single_preds.items():
            if base_col in {"Characteristics[Label]", "Characteristics[BiologicalReplicate]", "Comment[FractionIdentifier]"}:
                continue
            set_base_value(row, base_col, val, overwrite=True)

        # Static multi-slot study metadata, copy onto every row
        for base_col in STATIC_MULTI_SLOT_COLS:
            if base_col in multi_preds:
                set_base_values_across_slots(row, base_col, multi_preds[base_col])

        # Per-file filename hints
        file_meta = parse_filename_metadata_per_file(raw_file)
        for base_col, val in file_meta.items():
            if base_col in BASE_META_COLS:
                set_base_value(row, base_col, val, overwrite=True)

        rows.append(row)

    # Study-level label assignment
    if "Characteristics[Label]" in multi_preds:
        assign_round_robin_base(rows, "Characteristics[Label]", multi_preds["Characteristics[Label]"], overwrite_if_default_only=True)

    # Study-level biological replicate assignment
    if "Characteristics[BiologicalReplicate]" in multi_preds:
        rep_vals = sorted(multi_preds["Characteristics[BiologicalReplicate]"], key=_rep_sort_key)
        rep_vals = rep_vals[:max(1, min(len(rep_vals), len(rows)))]
        assign_round_robin_base(rows, "Characteristics[BiologicalReplicate]", rep_vals, overwrite_if_default_only=True)

    # Fraction fallback if file-level regex missed some but the study clearly has a fraction set
    frac_vals = file_hints.get("Comment[FractionIdentifier]", [])
    if frac_vals:
        all_blank_frac = all(
            get_base_value_from_row(r, "Comment[FractionIdentifier]") in [None, DEFAULT_FILL]
            for r in rows
        )
        if all_blank_frac:
            assign_round_robin_base(rows, "Comment[FractionIdentifier]", frac_vals, overwrite_if_default_only=True)

    pred_df = pd.DataFrame(rows)

    for col in SUB_COLUMNS:
        if col not in pred_df.columns:
            pred_df[col] = DEFAULT_FILL

    pred_df = pred_df[SUB_COLUMNS]

    if debug:
        print("=== single_preds ===")
        for k, v in sorted(single_preds.items()):
            print(k, "->", v)

        print("\n=== multi_preds ===")
        for k, v in sorted(multi_preds.items()):
            print(k, "->", v)

        print("\n=== top candidate scores ===")
        for base_col in sorted(cands.keys()):
            ranked = sorted(cands[base_col].items(), key=lambda kv: (-kv[1], kv[0]))[:5]
            if ranked:
                print(base_col, ":", ranked)

        print("\n=== neighbors ===")
        display(neighbors_df)

    return pred_df

# smoke test
pred_v5_example = baseline_predict_pxd_v5(TRAIN_PXDS[0], split="train", debug=False)
print("Example prediction shape:", pred_v5_example.shape)
display(pred_v5_example.head())

Example prediction shape: (6, 81)


,ID,PXD,Raw Data File,Characteristics[Age],Characteristics[AlkylationReagent],Characteristics[AnatomicSiteTumor],Characteristics[AncestryCategory],Characteristics[BMI],Characteristics[Bait],Characteristics[BiologicalReplicate],...,FactorValue[Bait],FactorValue[CellPart],FactorValue[Compound],FactorValue[ConcentrationOfCompound].1,FactorValue[Disease],FactorValue[FractionIdentifier],FactorValue[GeneticModification],FactorValue[Temperature],FactorValue[Treatment],Usage
0,1,PXD000070,OTPf-IMACDDNL_2010Mar9-01.raw,Not Applicable,Iodoacetamide,Not Applicable,Not Applicable,Not Applicable,Not Applicable,1,...,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable
1,2,PXD000070,OTPf-IMACDT2010Mar11-01.raw,Not Applicable,Iodoacetamide,Not Applicable,Not Applicable,Not Applicable,Not Applicable,1,...,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable
2,3,PXD000070,OTPf-IMACDT2010Mar11-02.raw,Not Applicable,Iodoacetamide,Not Applicable,Not Applicable,Not Applicable,Not Applicable,1,...,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable
3,4,PXD000070,OTPf-IMACDT2010Mar10-01.raw,Not Applicable,Iodoacetamide,Not Applicable,Not Applicable,Not Applicable,Not Applicable,1,...,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable
4,5,PXD000070,OTPf-IMACDDNL_2010Mar9-02.raw,Not Applicable,Iodoacetamide,Not Applicable,Not Applicable,Not Applicable,Not Applicable,1,...,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable


In [31]:
# ================================
# SAFE PATCH CELL A: V6 HELPERS
# - no monkeypatching
# - define NEW function names only
# ================================

import difflib

COLUMN_RETRIEVAL_SPECS_V6 = {
    "Characteristics[OrganismPart]": {"text_key": "sample_text",  "top_k": 8, "min_sim": 0.04, "min_share": 0.12, "multi": True},
    "Characteristics[Disease]":      {"text_key": "sample_text",  "top_k": 8, "min_sim": 0.04, "min_share": 0.12, "multi": True},
    "Characteristics[Modification]": {"text_key": "methods_text", "top_k": 8, "min_sim": 0.04, "min_share": 0.14, "multi": True},
    "Characteristics[Label]":        {"text_key": "full_text",    "top_k": 8, "min_sim": 0.04, "min_share": 0.14, "multi": True},
    "Characteristics[MaterialType]": {"text_key": "sample_text",  "top_k": 8, "min_sim": 0.04, "min_share": 0.20, "multi": False},
    "Characteristics[CellType]":     {"text_key": "sample_text",  "top_k": 8, "min_sim": 0.04, "min_share": 0.20, "multi": False},
}

def build_column_retrievers_v6():
    models = {}

    for base_col, spec in COLUMN_RETRIEVAL_SPECS_V6.items():
        pxds, texts, value_lists = [], [], []

        for rec in TRAIN_RECORDS:
            vals = rec["gold_values"].get(base_col, [])
            if not vals:
                vals = rec["gpt_values"].get(base_col, [])
            if not vals:
                continue

            txt = clean_text(rec.get(spec["text_key"], "") or rec.get("full_text", ""))
            if not txt:
                continue

            pxds.append(rec["pxd"])
            texts.append(txt)
            value_lists.append(vals)

        if len(texts) < 3:
            continue

        vec = TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=1,
            lowercase=True,
            strip_accents="unicode",
        )
        X = vec.fit_transform(texts)

        models[base_col] = {
            "pxds": pxds,
            "texts": texts,
            "value_lists": value_lists,
            "vectorizer": vec,
            "X": X,
            "spec": spec,
        }

    return models

COLUMN_RETRIEVER_MODELS_V6 = build_column_retrievers_v6()

def get_column_neighbors_v6(pub_json: dict, base_col: str, exclude_pxd: Optional[str] = None) -> pd.DataFrame:
    if base_col not in COLUMN_RETRIEVER_MODELS_V6:
        return pd.DataFrame(columns=["pxd", "sim", "vals"])

    model = COLUMN_RETRIEVER_MODELS_V6[base_col]
    spec = model["spec"]

    views = get_text_views(pub_json)
    qtxt = clean_text(views.get(spec["text_key"], "") or views.get("full_text", ""))
    if not qtxt:
        return pd.DataFrame(columns=["pxd", "sim", "vals"])

    qX = model["vectorizer"].transform([qtxt])
    sims = linear_kernel(qX, model["X"]).ravel()
    order = np.argsort(-sims)

    rows = []
    for idx in order:
        sim = float(sims[idx])
        pxd = model["pxds"][idx]

        if exclude_pxd is not None and pxd == exclude_pxd:
            continue
        if sim < spec["min_sim"]:
            continue

        rows.append({
            "pxd": pxd,
            "sim": sim,
            "vals": model["value_lists"][idx],
        })

        if len(rows) >= spec["top_k"]:
            break

    return pd.DataFrame(rows)

def add_column_specific_neighbor_candidates_v6(pub_json: dict, cands: Dict[str, Counter], exclude_pxd: Optional[str] = None):
    for base_col, spec in COLUMN_RETRIEVAL_SPECS_V6.items():
        ndf = get_column_neighbors_v6(pub_json, base_col, exclude_pxd=exclude_pxd)
        if len(ndf) == 0:
            continue

        weights = Counter()
        hit_count = Counter()
        total_weight = 0.0

        for _, row in ndf.iterrows():
            sim = float(row["sim"])
            vals = row["vals"]
            total_weight += sim

            for v in vals:
                vv = canonicalize_value(base_col, v)
                if vv is None:
                    continue
                weights[vv] += sim
                hit_count[vv] += 1

        if total_weight <= 0 or len(weights) == 0:
            continue

        ranked = []
        for v, w in weights.items():
            share = w / total_weight
            ranked.append((v, share, hit_count[v]))

        ranked.sort(key=lambda x: (-x[1], -x[2], x[0]))

        if spec["multi"]:
            max_values = MULTI_VALUE_COLS.get(base_col, {}).get("max_values", 4)
            for v, share, hits in ranked[:max_values]:
                if share >= spec["min_share"]:
                    add_candidate(cands, base_col, v, 0.95 + 1.85 * share + 0.10 * min(hits, 3), "col_neighbor_multi_v6")
        else:
            v, share, hits = ranked[0]
            if share >= spec["min_share"]:
                add_candidate(cands, base_col, v, 1.05 + 2.05 * share + 0.10 * min(hits, 3), "col_neighbor_single_v6")

def canonicalize_fraction_identifier_v6(value: str) -> Optional[str]:
    s = clean_text(value)
    if not s:
        return None

    norm = normalize_for_match(s)

    cmap = COLUMN_CANON_MAP.get("Comment[FractionIdentifier]", {})
    if norm in cmap:
        return cmap[norm]

    vocab = COLUMN_VOCAB.get("Comment[FractionIdentifier]", [])
    digits = re.findall(r"\d+", s)
    letters = re.findall(r"[A-Za-z]+", s)

    best = None
    best_score = -1.0

    for cand in vocab:
        nc = normalize_for_match(cand)
        score = 0.0

        if digits:
            if any(re.search(rf"(?<!\d)0*{d}(?!\d)", nc) for d in digits):
                score += 2.0

        if letters:
            joined = "".join(letters).lower()
            if joined and joined in nc.replace(" ", ""):
                score += 0.75

        score += difflib.SequenceMatcher(None, norm, nc).ratio()

        if score > best_score:
            best_score = score
            best = cand

    if best is not None and best_score >= 2.15:
        return best

    m = re.search(r"(?i)(?:frac|fraction|fx|f|band|slice|pool|offgel|scx|hph)[-_ ]*([A-Za-z]|\d{1,3})", s)
    if m:
        tok = m.group(1)
        if tok.isdigit():
            return f"F{int(tok)}"
        return f"F{tok.upper()}"

    m = re.search(r"(?<!\d)(\d{1,3})(?!\d)", s)
    if m:
        return f"F{int(m.group(1))}"

    return s

def parse_filename_metadata_per_file_v6(raw_file: str) -> Dict[str, str]:
    # start with the clean V5 parser
    out = parse_filename_metadata_per_file(raw_file).copy()

    name = Path(str(raw_file)).name
    stem = name.rsplit(".", 1)[0]

    frac_patterns = [
        r"(?i)(?:^|[_\-.])((?:frac|fraction|fx|f|band|slice|pool|offgel|scx|hph)[-_ ]*[A-Za-z]?\d{1,3}|(?:frac|fraction|fx|f|band|slice|pool|offgel|scx|hph)[-_ ]*[A-Za-z])(?:$|[_\-.])",
        r"(?i)(?:^|[_\-.])(F\d{1,3})(?:$|[_\-.])",
    ]

    for pat in frac_patterns:
        m = re.search(pat, stem)
        if m:
            frac_token = clean_text(m.group(1)).strip("_-. ")
            out["Comment[FractionIdentifier]"] = canonicalize_fraction_identifier_v6(frac_token)
            break

    # normalize biological replicate as clean integer-like string
    if "Characteristics[BiologicalReplicate]" in out:
        m = re.search(r"\d+", str(out["Characteristics[BiologicalReplicate]"]))
        if m:
            out["Characteristics[BiologicalReplicate]"] = str(int(m.group(0)))

    return out

def infer_study_hints_from_filenames_v6(raw_files: List[str]) -> Dict[str, List[str]]:
    counts = defaultdict(Counter)

    for rf in raw_files:
        md = parse_filename_metadata_per_file_v6(rf)
        for k, v in md.items():
            if v is None:
                continue
            counts[k][v] += 1

    out = {}
    for k, ctr in counts.items():
        out[k] = [v for v, _ in sorted(ctr.items(), key=lambda kv: (-kv[1], kv[0]))]
    return out

def collect_explicit_candidates_v6(pub_json: dict, raw_files: List[str]):
    # use the clean V5 extractor as a base, then add extra evidence
    cands, views, _ = collect_explicit_candidates(pub_json, raw_files)

    full_text = views["full_text"]
    methods_text = views["methods_text"] or views["core_text"]
    sampleish_text = " ".join([
        views["title"], views["abstract"], views["sample_text"],
        views["results_text"], views["caption_text"]
    ])

    file_hints = infer_study_hints_from_filenames_v6(raw_files)
    full_loose = normalize_for_loose_text(full_text + " " + methods_text)

    # stronger label phrases
    if re.search(r"\btandem mass tag(?:s)?\b", full_text, flags=re.I):
        add_candidate(cands, "Characteristics[Label]", "TMT", 2.6, "label_phrase_v6")
    if re.search(r"\bisobaric tags? for relative and absolute quantitation\b", full_text, flags=re.I):
        add_candidate(cands, "Characteristics[Label]", "iTRAQ", 2.6, "label_phrase_v6")

    # enrichment -> modification priors
    if any(x in full_loose for x in [" imac ", " tio2 ", " titanium dioxide ", " phosphotyrosine "]):
        add_candidate(cands, "Characteristics[Modification]", "phosphorylation", 3.0, "enrich_to_mod_v6")
    if any(x in full_loose for x in [" glyco ", " lectin ", " sialylated "]):
        add_candidate(cands, "Characteristics[Modification]", "glycosylation", 2.6, "enrich_to_mod_v6")
    if any(x in full_loose for x in [" acetyllysine ", " anti-acetyllysine ", " acetylated peptides "]):
        add_candidate(cands, "Characteristics[Modification]", "acetylation", 2.6, "enrich_to_mod_v6")

    # focused window rescans
    disease_windows = get_anchor_windows(sampleish_text, DISEASE_ANCHORS, radius=240)
    orgpart_windows = get_anchor_windows(sampleish_text, ORGPART_ANCHORS, radius=240)

    for v, cnt in ranked_vocab_mentions("Characteristics[Disease]", disease_windows, max_values=6):
        add_candidate(cands, "Characteristics[Disease]", v, 2.0 + 0.45 * cnt, "disease_window_vocab_v6")

    for v, cnt in ranked_vocab_mentions("Characteristics[OrganismPart]", orgpart_windows, max_values=8):
        add_candidate(cands, "Characteristics[OrganismPart]", v, 2.0 + 0.40 * cnt, "orgpart_window_vocab_v6")

    # filename-derived study hints using better fraction parsing
    for v in file_hints.get("Characteristics[Label]", []):
        add_candidate(cands, "Characteristics[Label]", v, 2.4, "filename_hint_v6")
    for v in file_hints.get("Characteristics[BiologicalReplicate]", []):
        add_candidate(cands, "Characteristics[BiologicalReplicate]", v, 2.4, "filename_hint_v6")
    for v in file_hints.get("Comment[FractionIdentifier]", []):
        add_candidate(cands, "Comment[FractionIdentifier]", v, 2.8, "filename_hint_v6")

    return cands, views, file_hints

print("Safe Patch Cell A loaded.")
print("Column-specific retrievers:", sorted(COLUMN_RETRIEVER_MODELS_V6.keys()))

Safe Patch Cell A loaded.
Column-specific retrievers: ['Characteristics[CellType]', 'Characteristics[Disease]', 'Characteristics[Label]', 'Characteristics[MaterialType]', 'Characteristics[Modification]', 'Characteristics[OrganismPart]']


In [32]:
# ================================
# SAFE PATCH CELL B: V6 EVIDENCE + PREDICTOR
# - again, no monkeypatching
# ================================

def build_prediction_evidence_v6(pub_json: dict, raw_files: List[str], exclude_pxd: Optional[str] = None):
    cands, views, file_hints = collect_explicit_candidates_v6(pub_json, raw_files)

    # column-specific retrieval first for weak high-value columns
    add_column_specific_neighbor_candidates_v6(pub_json, cands, exclude_pxd=exclude_pxd)

    # then global retrieval from V5
    neighbors_df = add_neighbor_candidates(pub_json, cands, exclude_pxd=exclude_pxd)

    single_preds = {}
    multi_preds = {}

    for base_col in BASE_META_COLS:
        if base_col in MULTI_VALUE_COLS:
            continue
        if base_col == "Comment[FractionIdentifier]":
            continue
        if base_col == "Characteristics[BiologicalReplicate]":
            continue

        min_sc = SINGLE_VALUE_MIN_SCORE.get(base_col, 1.6)
        val = select_best_value(cands, base_col, min_score=min_sc)
        if val is not None:
            single_preds[base_col] = val

    for base_col, cfg in MULTI_VALUE_COLS.items():
        vals = select_multi_values(
            cands,
            base_col,
            min_score=cfg["min_score"],
            max_values=cfg["max_values"],
        )
        if vals:
            multi_preds[base_col] = vals

    if "Characteristics[MaterialType]" not in single_preds:
        loose = normalize_for_loose_text(views["sample_text"] + " " + views["full_text"])
        if any(x in loose for x in [" plasma ", " serum ", " urine ", " saliva ", " blood "]):
            single_preds["Characteristics[MaterialType]"] = canonicalize_value("Characteristics[MaterialType]", "biofluid")
        elif "Characteristics[CellLine]" in single_preds:
            single_preds["Characteristics[MaterialType]"] = canonicalize_value("Characteristics[MaterialType]", "cell line")
        elif "Characteristics[OrganismPart]" in multi_preds or "Characteristics[OrganismPart]" in single_preds:
            single_preds["Characteristics[MaterialType]"] = canonicalize_value("Characteristics[MaterialType]", "tissue")

    return single_preds, multi_preds, cands, neighbors_df, file_hints, views

def baseline_predict_pxd_v6(pxd: str, split: str = "train", debug: bool = False) -> pd.DataFrame:
    pub_json = load_pub_json(pxd, split=split)
    raw_files = extract_raw_files(pub_json)

    if len(raw_files) == 0:
        raw_files = [f"{pxd}_file1.raw"]

    exclude = pxd if split == "train" else None

    single_preds, multi_preds, cands, neighbors_df, file_hints, views = build_prediction_evidence_v6(
        pub_json,
        raw_files,
        exclude_pxd=exclude,
    )

    rows = []
    for i, raw_file in enumerate(raw_files, start=1):
        row = make_blank_row()

        if "ID" in row:
            row["ID"] = i
        if "PXD" in row:
            row["PXD"] = pxd
        if "Raw Data File" in row:
            row["Raw Data File"] = raw_file
        if "Source Name" in row:
            row["Source Name"] = f"{pxd}_source_{i}"
        if "Assay Name" in row:
            row["Assay Name"] = Path(str(raw_file)).stem[:200]

        for base_col, val in single_preds.items():
            if base_col in {"Characteristics[Label]", "Characteristics[BiologicalReplicate]", "Comment[FractionIdentifier]"}:
                continue
            set_base_value(row, base_col, val, overwrite=True)

        for base_col in STATIC_MULTI_SLOT_COLS:
            if base_col in multi_preds:
                set_base_values_across_slots(row, base_col, multi_preds[base_col])

        file_meta = parse_filename_metadata_per_file_v6(raw_file)
        for base_col, val in file_meta.items():
            if base_col in BASE_META_COLS:
                set_base_value(row, base_col, val, overwrite=True)

        rows.append(row)

    if "Characteristics[Label]" in multi_preds:
        assign_round_robin_base(rows, "Characteristics[Label]", multi_preds["Characteristics[Label]"], overwrite_if_default_only=True)

    if "Characteristics[BiologicalReplicate]" in multi_preds:
        rep_vals = sorted(multi_preds["Characteristics[BiologicalReplicate]"], key=_rep_sort_key)
        rep_vals = rep_vals[:max(1, min(len(rep_vals), len(rows)))]
        assign_round_robin_base(rows, "Characteristics[BiologicalReplicate]", rep_vals, overwrite_if_default_only=True)

    frac_vals = file_hints.get("Comment[FractionIdentifier]", [])
    if frac_vals:
        all_blank_frac = all(
            get_base_value_from_row(r, "Comment[FractionIdentifier]") in [None, DEFAULT_FILL]
            for r in rows
        )
        if all_blank_frac:
            assign_round_robin_base(rows, "Comment[FractionIdentifier]", frac_vals, overwrite_if_default_only=True)

    pred_df = pd.DataFrame(rows)

    for col in SUB_COLUMNS:
        if col not in pred_df.columns:
            pred_df[col] = DEFAULT_FILL

    pred_df = pred_df[SUB_COLUMNS]

    if debug:
        print("=== single_preds ===")
        for k, v in sorted(single_preds.items()):
            print(k, "->", v)

        print("\n=== multi_preds ===")
        for k, v in sorted(multi_preds.items()):
            print(k, "->", v)

        print("\n=== top candidate scores ===")
        for base_col in sorted(cands.keys()):
            ranked = sorted(cands[base_col].items(), key=lambda kv: (-kv[1], kv[0]))[:5]
            if ranked:
                print(base_col, ":", ranked)

        print("\n=== neighbors ===")
        display(neighbors_df)

    return pred_df

# smoke test
pred_v6_example = baseline_predict_pxd_v6(TRAIN_PXDS[0], split="train", debug=False)
print("Example prediction shape:", pred_v6_example.shape)
display(pred_v6_example.head())

Example prediction shape: (6, 81)


,ID,PXD,Raw Data File,Characteristics[Age],Characteristics[AlkylationReagent],Characteristics[AnatomicSiteTumor],Characteristics[AncestryCategory],Characteristics[BMI],Characteristics[Bait],Characteristics[BiologicalReplicate],...,FactorValue[Bait],FactorValue[CellPart],FactorValue[Compound],FactorValue[ConcentrationOfCompound].1,FactorValue[Disease],FactorValue[FractionIdentifier],FactorValue[GeneticModification],FactorValue[Temperature],FactorValue[Treatment],Usage
0,1,PXD000070,OTPf-IMACDDNL_2010Mar9-01.raw,Not Applicable,Iodoacetamide,Not Applicable,Not Applicable,Not Applicable,Not Applicable,1,...,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable
1,2,PXD000070,OTPf-IMACDT2010Mar11-01.raw,Not Applicable,Iodoacetamide,Not Applicable,Not Applicable,Not Applicable,Not Applicable,1,...,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable
2,3,PXD000070,OTPf-IMACDT2010Mar11-02.raw,Not Applicable,Iodoacetamide,Not Applicable,Not Applicable,Not Applicable,Not Applicable,1,...,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable
3,4,PXD000070,OTPf-IMACDT2010Mar10-01.raw,Not Applicable,Iodoacetamide,Not Applicable,Not Applicable,Not Applicable,Not Applicable,1,...,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable
4,5,PXD000070,OTPf-IMACDDNL_2010Mar9-02.raw,Not Applicable,Iodoacetamide,Not Applicable,Not Applicable,Not Applicable,Not Applicable,1,...,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable


In [35]:
# ================================
# SAFE PATCH CELL C: V7 ALIAS LEXICON + FILENAME ROW RETRIEVERS
# ================================

FILENAME_TARGET_COLS_V7 = [
    "Comment[FractionIdentifier]",
    "Characteristics[BiologicalReplicate]",
    "Characteristics[Label]",
]

ALIAS_SCAN_COLS_V7 = [
    "Characteristics[OrganismPart]",
    "Characteristics[Disease]",
    "Characteristics[Modification]",
    "Characteristics[CellType]",
    "Characteristics[CellLine]",
    "Characteristics[MaterialType]",
]

GENERIC_ALIAS_BLOCKLIST_V7 = {
    "sample", "samples", "tissue", "tissues", "cell", "cells",
    "disease", "diseases", "healthy", "control", "controls",
    "normal", "adult", "male", "female", "tumor", "tumour"
}

def row_real_base_values_v7(row: pd.Series, base_col: str) -> List[str]:
    vals = []
    for c in row.index:
        if base_annotation(c) == base_col:
            vals.append(row[c])
    return unique_real_values_in_order(vals)

def first_row_base_value_v7(row: pd.Series, base_col: str) -> Optional[str]:
    vals = row_real_base_values_v7(row, base_col)
    return vals[0] if vals else None

def tokenize_filename_stem_v7(raw_file: str) -> List[str]:
    stem = Path(str(raw_file)).stem
    parts = re.split(r"[^A-Za-z0-9]+", stem)
    return [p for p in parts if p]

def build_filename_row_bank_v7() -> pd.DataFrame:
    rows = []

    for pxd in TRAIN_PXDS:
        df = load_gold_sdrf(pxd).copy()
        if "Raw Data File" not in df.columns:
            continue

        for _, r in df.iterrows():
            raw_file = clean_text(r.get("Raw Data File", ""))
            if not raw_file or not raw_file.lower().endswith(".raw"):
                continue

            stem = Path(raw_file).stem
            toks = tokenize_filename_stem_v7(raw_file)

            for base_col in FILENAME_TARGET_COLS_V7:
                val = first_row_base_value_v7(r, base_col)
                if not val:
                    continue

                rows.append({
                    "pxd": pxd,
                    "base_col": base_col,
                    "raw_file": Path(raw_file).name,
                    "stem": stem,
                    "tokens": toks,
                    "value": clean_text(val),
                })

    return pd.DataFrame(rows)

FILENAME_ROW_BANK_V7 = build_filename_row_bank_v7()

def build_filename_retrievers_v7():
    models = {}

    for base_col in FILENAME_TARGET_COLS_V7:
        sub = FILENAME_ROW_BANK_V7[FILENAME_ROW_BANK_V7["base_col"] == base_col].copy()
        if len(sub) < 10:
            continue

        texts = sub["stem"].astype(str).tolist()

        vec = TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(2, 5),
            min_df=1,
            lowercase=True,
            strip_accents="unicode",
        )
        X = vec.fit_transform(texts)

        models[base_col] = {
            "df": sub.reset_index(drop=True),
            "vectorizer": vec,
            "X": X,
        }

    return models

FILENAME_RETRIEVER_MODELS_V7 = build_filename_retrievers_v7()

def get_filename_neighbors_v7(raw_file: str, base_col: str, exclude_pxd: Optional[str] = None,
                              top_k: int = 20, min_sim: float = 0.08) -> pd.DataFrame:
    if base_col not in FILENAME_RETRIEVER_MODELS_V7:
        return pd.DataFrame(columns=["pxd", "sim", "value", "raw_file", "stem"])

    model = FILENAME_RETRIEVER_MODELS_V7[base_col]
    qstem = Path(str(raw_file)).stem

    qX = model["vectorizer"].transform([qstem])
    sims = linear_kernel(qX, model["X"]).ravel()
    order = np.argsort(-sims)

    rows = []
    for idx in order:
        sim = float(sims[idx])
        r = model["df"].iloc[idx]

        if exclude_pxd is not None and r["pxd"] == exclude_pxd:
            continue
        if sim < min_sim:
            continue

        rows.append({
            "pxd": r["pxd"],
            "sim": sim,
            "value": r["value"],
            "raw_file": r["raw_file"],
            "stem": r["stem"],
        })

        if len(rows) >= top_k:
            break

    return pd.DataFrame(rows)

def _digit_groups_v7(s: str) -> List[str]:
    return re.findall(r"\d+", str(s))

def best_filename_value_v7(raw_file: str, base_col: str, exclude_pxd: Optional[str] = None):
    ndf = get_filename_neighbors_v7(raw_file, base_col, exclude_pxd=exclude_pxd)
    if len(ndf) == 0:
        return None, 0.0, pd.DataFrame()

    qstem = Path(str(raw_file)).stem
    qdigits = _digit_groups_v7(qstem)

    weights = Counter()
    total_weight = 0.0

    for _, row in ndf.iterrows():
        sim = float(row["sim"])
        val = clean_text(row["value"])
        if not val:
            continue

        bonus = 0.0

        if base_col == "Comment[FractionIdentifier]":
            vdigits = _digit_groups_v7(val)
            if qdigits and vdigits and any(a == b for a in qdigits for b in vdigits):
                bonus += 0.35

        if base_col == "Characteristics[BiologicalReplicate]":
            vdigits = _digit_groups_v7(val)
            if qdigits and vdigits and any(a == b for a in qdigits for b in vdigits):
                bonus += 0.25

        total_weight += (sim + bonus)
        weights[val] += (sim + bonus)

    if total_weight <= 0 or len(weights) == 0:
        return None, 0.0, ndf

    ranked = sorted(weights.items(), key=lambda kv: (-kv[1], kv[0]))
    best_val, best_w = ranked[0]
    best_share = best_w / total_weight

    return best_val, best_share, ndf

def filename_row_predictions_v7(raw_file: str, exclude_pxd: Optional[str] = None) -> Dict[str, str]:
    out = {}

    # fraction: use retriever first, then fallback to the ORIGINAL parser (not the v6 over-canonicalized one)
    frac_val, frac_share, _ = best_filename_value_v7(raw_file, "Comment[FractionIdentifier]", exclude_pxd=exclude_pxd)
    if frac_val is not None and frac_share >= 0.22:
        out["Comment[FractionIdentifier]"] = frac_val
    else:
        old_meta = parse_filename_metadata_per_file(raw_file)
        if "Comment[FractionIdentifier]" in old_meta and clean_text(old_meta["Comment[FractionIdentifier]"]):
            out["Comment[FractionIdentifier]"] = old_meta["Comment[FractionIdentifier]"]

    # biorep
    br_val, br_share, _ = best_filename_value_v7(raw_file, "Characteristics[BiologicalReplicate]", exclude_pxd=exclude_pxd)
    if br_val is not None and br_share >= 0.22:
        m = re.search(r"\d+", str(br_val))
        out["Characteristics[BiologicalReplicate]"] = str(int(m.group(0))) if m else clean_text(br_val)

    # label
    lab_val, lab_share, _ = best_filename_value_v7(raw_file, "Characteristics[Label]", exclude_pxd=exclude_pxd)
    if lab_val is not None and lab_share >= 0.18:
        out["Characteristics[Label]"] = lab_val

    # merge the safer V6 parser for the other file-level things
    meta_v6 = parse_filename_metadata_per_file_v6(raw_file)
    for k, v in meta_v6.items():
        if k not in out and clean_text(v):
            out[k] = v

    return out

# -------- alias lexicon mined from train vocab --------

def derive_aliases_from_value_v7(v: str) -> List[str]:
    s = clean_text(v)
    if not s:
        return []

    aliases = set()
    aliases.add(s)

    # remove ontology-like IDs
    s_no_id = re.sub(r"^[A-Za-z_]+:\d+\s*", "", s).strip()
    if s_no_id:
        aliases.add(s_no_id)

    # inside / outside parentheses
    paren_parts = re.findall(r"\(([^)]{2,})\)", s)
    outside = re.sub(r"\([^)]*\)", " ", s).strip()
    if outside:
        aliases.add(clean_text(outside))
    for p in paren_parts:
        aliases.add(clean_text(p))

    # split composite values
    for piece in re.split(r"[;/|,]", s):
        piece = clean_text(piece)
        if piece:
            aliases.add(piece)

    out = []
    for a in aliases:
        na = normalize_for_match(a)
        if not na:
            continue
        if na in GENERIC_ALIAS_BLOCKLIST_V7:
            continue
        if len(na) < 4 and not re.search(r"\d", na):
            continue
        out.append(a)

    return list(dict.fromkeys(out))

def build_alias_lexicon_v7():
    lex = {}

    for base_col in ALIAS_SCAN_COLS_V7:
        alias_to_canon = defaultdict(set)

        for canon in COLUMN_VOCAB.get(base_col, []):
            for alias in derive_aliases_from_value_v7(canon):
                alias_to_canon[normalize_for_match(alias)].add(canon)

        clean_map = {}
        for alias_norm, canons in alias_to_canon.items():
            if len(canons) == 1:
                clean_map[alias_norm] = list(canons)[0]

        lex[base_col] = clean_map

    return lex

COLUMN_ALIAS_LEXICON_V7 = build_alias_lexicon_v7()

def alias_mentions_in_text_v7(base_col: str, text: str, max_hits: int = 12) -> List[str]:
    amap = COLUMN_ALIAS_LEXICON_V7.get(base_col, {})
    if not amap:
        return []

    loose = normalize_for_loose_text(text)
    ranked_aliases = sorted(amap.keys(), key=lambda x: (-len(x), x))

    out = []
    seen = set()

    for alias_norm in ranked_aliases:
        if f" {alias_norm} " in loose:
            canon = amap[alias_norm]
            if canon not in seen:
                seen.add(canon)
                out.append(canon)
        if len(out) >= max_hits:
            break

    return out

print("Safe Patch Cell C loaded.")
print("Filename row bank shape:", FILENAME_ROW_BANK_V7.shape)
print("Filename retriever cols:", sorted(FILENAME_RETRIEVER_MODELS_V7.keys()))
print("Alias lexicon cols:", sorted(COLUMN_ALIAS_LEXICON_V7.keys()))

Safe Patch Cell C loaded.
Filename row bank shape: (106033, 6)
Filename retriever cols: ['Characteristics[BiologicalReplicate]', 'Characteristics[Label]', 'Comment[FractionIdentifier]']
Alias lexicon cols: ['Characteristics[CellLine]', 'Characteristics[CellType]', 'Characteristics[Disease]', 'Characteristics[MaterialType]', 'Characteristics[Modification]', 'Characteristics[OrganismPart]']


In [36]:
# ================================
# SAFE PATCH CELL D: V7 EVIDENCE + PREDICTOR
# ================================

def collect_explicit_candidates_v7(pub_json: dict, raw_files: List[str]):
    cands, views, file_hints_v6 = collect_explicit_candidates_v6(pub_json, raw_files)

    sampleish_text = " ".join([
        views["title"], views["abstract"], views["sample_text"],
        views["results_text"], views["caption_text"]
    ])
    methods_text = views["methods_text"] or views["core_text"]
    full_text = views["full_text"]

    # alias-based evidence from train vocab
    for base_col, text, score in [
        ("Characteristics[OrganismPart]", sampleish_text, 2.3),
        ("Characteristics[Disease]", sampleish_text, 2.2),
        ("Characteristics[Modification]", methods_text + " " + full_text, 2.2),
        ("Characteristics[CellType]", sampleish_text, 1.8),
        ("Characteristics[CellLine]", sampleish_text, 1.9),
        ("Characteristics[MaterialType]", sampleish_text, 1.8),
    ]:
        for v in alias_mentions_in_text_v7(base_col, text, max_hits=10):
            add_candidate(cands, base_col, v, score, "alias_scan_v7")

    # focused windows with alias scan too
    disease_windows = get_anchor_windows(sampleish_text, DISEASE_ANCHORS, radius=240)
    orgpart_windows = get_anchor_windows(sampleish_text, ORGPART_ANCHORS, radius=240)

    for w in disease_windows:
        for v in alias_mentions_in_text_v7("Characteristics[Disease]", w, max_hits=6):
            add_candidate(cands, "Characteristics[Disease]", v, 1.6, "alias_window_v7")

    for w in orgpart_windows:
        for v in alias_mentions_in_text_v7("Characteristics[OrganismPart]", w, max_hits=8):
            add_candidate(cands, "Characteristics[OrganismPart]", v, 1.6, "alias_window_v7")

    # keep file hints from the better filename logic
    file_hints = infer_study_hints_from_filenames_v6(raw_files)

    for v in file_hints.get("Characteristics[Label]", []):
        add_candidate(cands, "Characteristics[Label]", v, 2.2, "filehint_v7")

    for v in file_hints.get("Characteristics[BiologicalReplicate]", []):
        add_candidate(cands, "Characteristics[BiologicalReplicate]", v, 2.2, "filehint_v7")

    # IMPORTANT: do not inject fraction here as a study-level candidate.
    # It should stay row-level.

    return cands, views, file_hints

def build_prediction_evidence_v7(pub_json: dict, raw_files: List[str], exclude_pxd: Optional[str] = None):
    cands, views, file_hints = collect_explicit_candidates_v7(pub_json, raw_files)

    # column-specific retrieval from V6
    add_column_specific_neighbor_candidates_v6(pub_json, cands, exclude_pxd=exclude_pxd)

    # global retrieval from V5/V6
    neighbors_df = add_neighbor_candidates(pub_json, cands, exclude_pxd=exclude_pxd)

    single_preds = {}
    multi_preds = {}

    for base_col in BASE_META_COLS:
        if base_col in MULTI_VALUE_COLS:
            continue
        if base_col == "Comment[FractionIdentifier]":
            continue
        if base_col == "Characteristics[BiologicalReplicate]":
            continue

        min_sc = SINGLE_VALUE_MIN_SCORE.get(base_col, 1.6)
        val = select_best_value(cands, base_col, min_score=min_sc)
        if val is not None:
            single_preds[base_col] = val

    for base_col, cfg in MULTI_VALUE_COLS.items():
        vals = select_multi_values(
            cands,
            base_col,
            min_score=cfg["min_score"],
            max_values=cfg["max_values"],
        )
        if vals:
            multi_preds[base_col] = vals

    if "Characteristics[MaterialType]" not in single_preds:
        loose = normalize_for_loose_text(views["sample_text"] + " " + views["full_text"])
        if any(x in loose for x in [" plasma ", " serum ", " urine ", " saliva ", " blood "]):
            single_preds["Characteristics[MaterialType]"] = canonicalize_value("Characteristics[MaterialType]", "biofluid")
        elif "Characteristics[CellLine]" in single_preds:
            single_preds["Characteristics[MaterialType]"] = canonicalize_value("Characteristics[MaterialType]", "cell line")
        elif "Characteristics[OrganismPart]" in multi_preds or "Characteristics[OrganismPart]" in single_preds:
            single_preds["Characteristics[MaterialType]"] = canonicalize_value("Characteristics[MaterialType]", "tissue")

    return single_preds, multi_preds, cands, neighbors_df, file_hints, views

def baseline_predict_pxd_v7(pxd: str, split: str = "train", debug: bool = False) -> pd.DataFrame:
    pub_json = load_pub_json(pxd, split=split)
    raw_files = extract_raw_files(pub_json)

    if len(raw_files) == 0:
        raw_files = [f"{pxd}_file1.raw"]

    exclude = pxd if split == "train" else None

    single_preds, multi_preds, cands, neighbors_df, file_hints, views = build_prediction_evidence_v7(
        pub_json,
        raw_files,
        exclude_pxd=exclude,
    )

    rows = []
    for i, raw_file in enumerate(raw_files, start=1):
        row = make_blank_row()

        if "ID" in row:
            row["ID"] = i
        if "PXD" in row:
            row["PXD"] = pxd
        if "Raw Data File" in row:
            row["Raw Data File"] = raw_file
        if "Source Name" in row:
            row["Source Name"] = f"{pxd}_source_{i}"
        if "Assay Name" in row:
            row["Assay Name"] = Path(str(raw_file)).stem[:200]

        # global single-value metadata
        for base_col, val in single_preds.items():
            if base_col in {"Characteristics[Label]", "Characteristics[BiologicalReplicate]", "Comment[FractionIdentifier]"}:
                continue
            set_base_value(row, base_col, val, overwrite=True)

        # static multi-slot study metadata
        for base_col in STATIC_MULTI_SLOT_COLS:
            if base_col in multi_preds:
                set_base_values_across_slots(row, base_col, multi_preds[base_col])

        # row-level filename evidence
        file_meta = filename_row_predictions_v7(raw_file, exclude_pxd=exclude)
        for base_col, val in file_meta.items():
            if base_col in BASE_META_COLS:
                set_base_value(row, base_col, val, overwrite=True)

        rows.append(row)

    # study-level label fill only for blanks
    if "Characteristics[Label]" in multi_preds:
        assign_round_robin_base(rows, "Characteristics[Label]", multi_preds["Characteristics[Label]"], overwrite_if_default_only=True)

    # study-level biorep fill only for blanks
    if "Characteristics[BiologicalReplicate]" in multi_preds:
        rep_vals = sorted(multi_preds["Characteristics[BiologicalReplicate]"], key=_rep_sort_key)
        rep_vals = rep_vals[:max(1, min(len(rep_vals), len(rows)))]
        assign_round_robin_base(rows, "Characteristics[BiologicalReplicate]", rep_vals, overwrite_if_default_only=True)

    # fraction fallback only if every row is blank
    frac_vals = file_hints.get("Comment[FractionIdentifier]", [])
    if frac_vals:
        all_blank_frac = all(
            get_base_value_from_row(r, "Comment[FractionIdentifier]") in [None, DEFAULT_FILL]
            for r in rows
        )
        if all_blank_frac:
            assign_round_robin_base(rows, "Comment[FractionIdentifier]", frac_vals, overwrite_if_default_only=True)

    pred_df = pd.DataFrame(rows)

    for col in SUB_COLUMNS:
        if col not in pred_df.columns:
            pred_df[col] = DEFAULT_FILL

    pred_df = pred_df[SUB_COLUMNS]

    if debug:
        print("=== single_preds ===")
        for k, v in sorted(single_preds.items()):
            print(k, "->", v)

        print("\n=== multi_preds ===")
        for k, v in sorted(multi_preds.items()):
            print(k, "->", v)

        print("\n=== top candidate scores ===")
        for base_col in sorted(cands.keys()):
            ranked = sorted(cands[base_col].items(), key=lambda kv: (-kv[1], kv[0]))[:5]
            if ranked:
                print(base_col, ":", ranked)

        print("\n=== neighbors ===")
        display(neighbors_df)

    return pred_df

# smoke test
pred_v7_example = baseline_predict_pxd_v7(TRAIN_PXDS[0], split="train", debug=False)
print("Example prediction shape:", pred_v7_example.shape)
display(pred_v7_example.head())

Example prediction shape: (6, 81)


,ID,PXD,Raw Data File,Characteristics[Age],Characteristics[AlkylationReagent],Characteristics[AnatomicSiteTumor],Characteristics[AncestryCategory],Characteristics[BMI],Characteristics[Bait],Characteristics[BiologicalReplicate],...,FactorValue[Bait],FactorValue[CellPart],FactorValue[Compound],FactorValue[ConcentrationOfCompound].1,FactorValue[Disease],FactorValue[FractionIdentifier],FactorValue[GeneticModification],FactorValue[Temperature],FactorValue[Treatment],Usage
0,1,PXD000070,OTPf-IMACDDNL_2010Mar9-01.raw,Not Applicable,Iodoacetamide,Not Applicable,Not Applicable,Not Applicable,Not Applicable,1,...,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable
1,2,PXD000070,OTPf-IMACDT2010Mar11-01.raw,Not Applicable,Iodoacetamide,Not Applicable,Not Applicable,Not Applicable,Not Applicable,1,...,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable
2,3,PXD000070,OTPf-IMACDT2010Mar11-02.raw,Not Applicable,Iodoacetamide,Not Applicable,Not Applicable,Not Applicable,Not Applicable,1,...,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable
3,4,PXD000070,OTPf-IMACDT2010Mar10-01.raw,Not Applicable,Iodoacetamide,Not Applicable,Not Applicable,Not Applicable,Not Applicable,1,...,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable
4,5,PXD000070,OTPf-IMACDDNL_2010Mar9-02.raw,Not Applicable,Iodoacetamide,Not Applicable,Not Applicable,Not Applicable,Not Applicable,1,...,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable


In [33]:
import difflib

class ParticipantVisibleError(Exception):
    pass

def official_load_sdrf(sdrf_df: pd.DataFrame) -> Dict[str, Dict[str, List[str]]]:

    sdrf_dict: Dict[str, Dict[str, List[str]]] = {}
    for pxd, pxd_df in sdrf_df.groupby("PXD"):
        sdrf_dict[pxd] = {}
        for col in pxd_df.columns:
            if col in ["Raw Data File", "Usage", "PXD"]:
                continue

            uniq = pd.Series(pxd_df[col]).dropna().astype(str).unique().tolist()

            if uniq == ["Not Applicable"]:
                continue

            values: List[str] = []
            for v in uniq:
                if "NT=" in v:
                    parts = [r for r in v.split(";") if "NT=" in r]
                    values.append(parts[0].replace("NT=", "").strip() if parts else v.strip())
                else:
                    values.append(v.strip())

            if "." in col:
                col = col.split(".")[0].strip()

            if col in sdrf_dict[pxd]:
                sdrf_dict[pxd][col] += values
            else:
                sdrf_dict[pxd][col] = values

    return sdrf_dict

def _string_similarity(a: str, b: str) -> float:
    return difflib.SequenceMatcher(None, a or "", b or "").ratio()

def official_harmonize_and_evaluate(
    A: Dict[str, Dict[str, List[str]]],
    B: Dict[str, Dict[str, List[str]]],
    threshold: float = 0.80,
    CompleteAbsence: float = float("nan"),
) -> Tuple[Dict[str, Dict[str, List[int]]], Dict[str, Dict[str, List[int]]], pd.DataFrame]:

    from sklearn.metrics import precision_score, recall_score, f1_score

    eval_metrics = {
        "pxd": [],
        "AnnotationType": [],
        "precision": [],
        "recall": [],
        "f1": [],
        "jacc": [],
    }

    harmonized_A: Dict[str, Dict[str, List[int]]] = {}
    harmonized_B: Dict[str, Dict[str, List[int]]] = {}

    common_pubs = set(A) & set(B)
    for pub in common_pubs:
        harmonized_A[pub], harmonized_B[pub] = {}, {}

        for category in set(A[pub]):
            vals_A = A[pub][category]
            vals_B = B[pub].get(category, [])

            all_vals = vals_A + [v for v in vals_B if v not in vals_A]

            if len(vals_A) == 0 and len(vals_B) == 0:
                harmA, harmB = [], []

            elif len(all_vals) == 1:
                str2cid = {all_vals[0]: 0}
                harmA = [str2cid[s] for s in vals_A]
                harmB = [str2cid[s] for s in vals_B]

            else:
                N = len(all_vals)
                dist = np.zeros((N, N), dtype=float)

                for i in range(N):
                    for j in range(i + 1, N):
                        sim = _string_similarity(all_vals[i], all_vals[j])
                        d = 1.0 - sim
                        dist[i, j] = d
                        dist[j, i] = d

                clusterer = AgglomerativeClustering(
                    n_clusters=None,
                    metric="precomputed",
                    linkage="average",
                    distance_threshold=1.0 - threshold,
                )
                labels = clusterer.fit_predict(dist)
                str2cid = {s: int(labels[i]) for i, s in enumerate(all_vals)}
                harmA = [str2cid[s] for s in vals_A]
                harmB = [str2cid[s] for s in vals_B]

            harmonized_A[pub][category] = harmA
            harmonized_B[pub][category] = harmB

            uniq = sorted(set(harmA) | set(harmB))
            if not uniq:
                p = r = f = CompleteAbsence
                j = 1.0
            else:
                y_true = [1 if u in harmA else 0 for u in uniq]
                y_pred = [1 if u in harmB else 0 for u in uniq]

                p = precision_score(y_true, y_pred, average="macro", zero_division=0)
                r = recall_score(y_true, y_pred, average="macro", zero_division=0)
                f = f1_score(y_true, y_pred, average="macro", zero_division=0)

                setA, setB = set(harmA), set(harmB)
                j = 1.0 if (not setA and not setB) else len(setA & setB) / len(setA | setB)

            eval_metrics["pxd"].append(pub)
            eval_metrics["AnnotationType"].append(category)
            eval_metrics["precision"].append(p)
            eval_metrics["recall"].append(r)
            eval_metrics["f1"].append(f)
            eval_metrics["jacc"].append(j)

    return harmonized_A, harmonized_B, pd.DataFrame(eval_metrics)

def official_score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str = "ID") -> float:
    if row_id_column_name and row_id_column_name in solution.columns:
        solution = solution.drop(columns=[row_id_column_name])
    if row_id_column_name and row_id_column_name in submission.columns:
        submission = submission.drop(columns=[row_id_column_name])

    if "PXD" not in solution.columns or "PXD" not in submission.columns:
        raise ParticipantVisibleError("Both solution and submission must include a 'PXD' column.")

    sol = official_load_sdrf(solution)
    sub = official_load_sdrf(submission)
    _, _, eval_df = official_harmonize_and_evaluate(sol, sub, threshold=0.80)

    vals = eval_df["f1"].dropna()
    return float(vals.mean()) if not vals.empty else 0.0

In [37]:
# ================================
# CELL 7: OFFICIAL TRAIN EVALUATION FOR V4
# ================================

def build_train_solution_df(train_pxds: List[str]) -> pd.DataFrame:
    parts = []
    for pxd in train_pxds:
        gold = load_gold_sdrf(pxd).copy()

        for col in SUB_COLUMNS:
            if col not in gold.columns:
                gold[col] = DEFAULT_FILL

        gold = gold[SUB_COLUMNS]
        parts.append(gold)

    return pd.concat(parts, ignore_index=True)

def build_train_submission_df(train_pxds: List[str], predictor_fn) -> pd.DataFrame:
    parts = []
    failed = []

    for pxd in train_pxds:
        try:
            pred = predictor_fn(pxd, split="train").copy()
            gold = load_gold_sdrf(pxd).copy()

            for col in SUB_COLUMNS:
                if col not in pred.columns:
                    pred[col] = DEFAULT_FILL

            pred = pred[SUB_COLUMNS]
            pred = fit_prediction_to_target_rows(pred, len(gold))

            if "PXD" in pred.columns:
                pred["PXD"] = pxd

            parts.append(pred)

        except Exception as e:
            failed.append((pxd, str(e)))

    if failed:
        print("FAILED PXDs:")
        for x in failed[:20]:
            print(x)

    out = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=SUB_COLUMNS)

    if "ID" in out.columns:
        out["ID"] = np.arange(1, len(out) + 1)

    return out

def official_eval_breakdown(solution_df: pd.DataFrame, submission_df: pd.DataFrame, row_id_column_name: str = "ID"):
    sol = solution_df.copy()
    sub = submission_df.copy()

    if row_id_column_name and row_id_column_name in sol.columns:
        sol = sol.drop(columns=[row_id_column_name])
    if row_id_column_name and row_id_column_name in sub.columns:
        sub = sub.drop(columns=[row_id_column_name])

    sol_dict = official_load_sdrf(sol)
    sub_dict = official_load_sdrf(sub)

    _, _, eval_df = official_harmonize_and_evaluate(sol_dict, sub_dict, threshold=0.80)

    by_col = (
        eval_df.groupby("AnnotationType", as_index=False)[["precision", "recall", "f1", "jacc"]]
        .mean()
        .sort_values("f1", ascending=False)
        .reset_index(drop=True)
    )

    by_col["n_pxd"] = (
        eval_df.groupby("AnnotationType")["pxd"]
        .nunique()
        .reindex(by_col["AnnotationType"])
        .values
    )

    return eval_df, by_col

gold_train_df = build_train_solution_df(TRAIN_PXDS)
pred_train_v7_df = build_train_submission_df(TRAIN_PXDS, baseline_predict_pxd_v7)

train_score_v4 = official_score(
    solution=gold_train_df.copy(),
    submission=pred_train_v7_df.copy(),
    row_id_column_name="ID",
)

eval_df_v4, by_col_v4 = official_eval_breakdown(
    gold_train_df,
    pred_train_v7_df,
    row_id_column_name="ID",
)

print("V4 official train score:", train_score_v4)
display(by_col_v4.head(60))

KeyboardInterrupt: 

V5 official train score: 0.4392898107928591

AnnotationType	precision	recall	f1	jacc	n_pxd
0	Characteristics[Organism]	0.882353	0.875613	0.876906	0.878676	102
1	Characteristics[CleavageAgent]	0.852941	0.836601	0.841270	0.849673	102
2	Comment[MS2MassAnalyzer]	0.785714	0.773810	0.777778	0.785714	21
3	Comment[FragmentationMethod]	0.782609	0.757246	0.764493	0.775362	46
4	Comment[Instrument]	0.730392	0.722222	0.724673	0.728758	102
5	Characteristics[BiologicalReplicate]	0.689089	0.679624	0.650209	0.706499	97
6	Comment[PrecursorMassTolerance]	0.594595	0.594595	0.594595	0.594595	74
7	Characteristics[ReductionReagent]	0.500000	0.500000	0.500000	0.500000	2
8	Characteristics[Label]	0.411356	0.517157	0.443900	0.505637	102
9	Comment[FragmentMassTolerance]	0.416667	0.416667	0.416667	0.416667	72
10	Characteristics[CellType]	0.381579	0.345395	0.356140	0.375000	38
11	Characteristics[MaterialType]	0.355932	0.331780	0.336723	0.341525	59
12	Comment[NumberOfMissedCleavages]	0.333333	0.333333	0.333333	0.333333	3
13	Characteristics[Modification]	0.288082	0.346416	0.300846	0.411257	93
14	Characteristics[OrganismPart]	0.297619	0.265073	0.271716	0.280146	84
15	Characteristics[CellLine]	0.326923	0.236386	0.254487	0.280464	26
16	Comment[FractionIdentifier]	0.239663	0.247551	0.233380	0.259885	99
17	Characteristics[Disease]	0.246753	0.211629	0.221861	0.241440	77
18	Comment[EnrichmentMethod]	0.222222	0.166667	0.185185	0.222222	9
19	Characteristics[DevelopmentalStage]	0.142857	0.142857	0.142857	0.142857	28
20	Characteristics[Sex]	0.065217	0.038043	0.045290	0.054348	46
21	Characteristics[Age]	0.027027	0.027027	0.027027	0.027027	37
22	Comment[FractionationMethod]	0.031250	0.015625	0.020833	0.031250	16
23	Characteristics[AlkylationReagent]	0.000000	0.000000	0.000000	0.000000	2
24	Characteristics[AncestryCategory]	0.000000	0.000000	0.000000	0.000000	17
25	Characteristics[Bait]	0.000000	0.000000	0.000000	0.000000	1
26	Characteristics[BMI]	0.000000	0.000000	0.000000	0.000000	1
27	Characteristics[Compound]	0.000000	0.000000	0.000000	0.000000	3
28	Characteristics[CellPart]	0.000000	0.000000	0.000000	0.000000	1
29	Characteristics[SpikedCompound]	0.000000	0.000000	0.000000	0.000000	3
30	Characteristics[Staining]	0.000000	0.000000	0.000000	0.000000	1
31	Characteristics[PooledSample]	0.000000	0.000000	0.000000	0.000000	1
32	Characteristics[Depletion]	0.000000	0.000000	0.000000	0.000000	6
33	Characteristics[GrowthRate]	0.000000	0.000000	0.000000	0.000000	1
34	Characteristics[SamplingTime]	0.000000	0.000000	0.000000	0.000000	2
35	Characteristics[Strain]	0.000000	0.000000	0.000000	0.000000	3
36	Comment[CollisionEnergy]	0.000000	0.000000	0.000000	0.000000	22
37	Characteristics[TumorSize]	0.000000	0.000000	0.000000	0.000000	1
38	Characteristics[Treatment]	0.000000	0.000000	0.000000	0.000000	9
39	Characteristics[Temperature]	0.000000	0.000000	0.000000	0.000000	1
40	Characteristics[SyntheticPeptide]	0.000000	0.000000	0.000000	0.000000	3
41	Characteristics[Time]	0.000000	0.000000	0.000000	0.000000	2
42	Comment[Separation]	0.000000	0.000000	0.000000	0.000000	8


In [ ]:
# ================================
# CELL 8: BUILD TEST SUBMISSION FOR V4
# ================================

sample_sub = pd.read_csv(SAMPLE_SUB_PATH).copy()

submission_parts = []

for pxd, template_df in sample_sub.groupby("PXD", sort=False):
    target_n = len(template_df)

    pred_df = baseline_predict_pxd_v7(pxd, split="test").copy()

    for col in SUB_COLUMNS:
        if col not in pred_df.columns:
            pred_df[col] = DEFAULT_FILL
    pred_df = pred_df[SUB_COLUMNS]

    pred_df = fit_prediction_to_target_rows(pred_df, target_n)

    if "ID" in pred_df.columns and "ID" in template_df.columns:
        pred_df["ID"] = template_df["ID"].values

    pred_df["PXD"] = template_df["PXD"].values

    submission_parts.append(pred_df)

submission_v7 = pd.concat(submission_parts, ignore_index=True)

for col in SUB_COLUMNS:
    if col not in submission_v7.columns:
        submission_v7[col] = DEFAULT_FILL
submission_v7 = submission_v7[SUB_COLUMNS]

print("submission_v7 shape:", submission_v7.shape)
print("sample_sub shape   :", sample_sub.shape)
print("unique IDs         :", submission_v7["ID"].nunique(), "out of", len(submission_v7))

count_check = (
    sample_sub.groupby("PXD").size().reset_index(name="sample_rows")
    .merge(submission_v7.groupby("PXD").size().reset_index(name="pred_rows"), on="PXD", how="left")
)
count_check["diff"] = count_check["pred_rows"] - count_check["sample_rows"]
display(count_check)

submission_v7.to_csv("submission_v7.csv", index=False)
print("saved -> submission_v7.csv")

submission_v5 shape: (1659, 81)
sample_sub shape   : (1659, 81)
unique IDs         : 1659 out of 1659


,PXD,sample_rows,pred_rows,diff
0,PXD004010,10,10,0
1,PXD016436,18,18,0
2,PXD019519,6,6,0
3,PXD025663,12,12,0
4,PXD040582,24,24,0
5,PXD050621,9,9,0
6,PXD061009,2,2,0
7,PXD061090,6,6,0
8,PXD061136,2,2,0
9,PXD061195,1376,1376,0


saved -> submission_v5.csv


Achieved a score of 0.15876 on the leaderboard which isn't great.